<div style="background: linear-gradient(135deg, #0d1117 0%, #161b22 50%, #0d1117 100%); border: 2px solid #30363d; border-radius: 12px; padding: 30px; margin-bottom: 20px;">

<h1 style="color: #f0f6fc; text-align: center; font-size: 2em; margin: 0 0 12px 0;">
  🔐 블록 암호화 실험 — ResNet50 기반 폐암 분류 (에지 컴퓨팅)
</h1>
<p style="color: #8b949e; text-align: center; font-size: 1.05em; margin: 0 0 20px 0;">
  Edge Computing · Block Encryption · Privacy-Preserving Lung Cancer Classification
</p>
</div>

---

## 📋 실험 목차

| # | 셀 | 내용 |
|---|---|---|
| 1 | Cell 1 | 라이브러리 임포트 |
| 2 | Cell 2 | 한글 폰트 설정 |
| 3 | **Cell 3** | **GPU 설정 & 모니터링 대시보드** |
| 4 | Cell 4 | 실험 설정 (하이퍼파라미터) |
| 5 | Cell 5 | 블록 암호화 함수 |
| 6 | Cell 6 | 오버샘플링 |
| 7 | Cell 7 | Train/Test 분리 |
| 8 | Cell 8 | tf.data 파이프라인 |
| 9 | Cell 9 | 학습 기법 (Focal Loss / Cosine LR / SWA) |
| 10 | Cell 10 | 모델 빌드 — 에지/클라우드 분리, end-to-end 학습 |
| 11 | **Cell 10b** | **학습 히스토리 기록 & 과적합/과소적합 진단** |
| 12 | Cell 11 | 단일 블록 실험 함수 |
| 13 | Cell 12 | 전체 실험 루프 |
| 14 | Cell 13 | 결과 표 출력 |
| 15 | **Cell 13b** | **과적합/과소적합 상태 자동 진단 표** |
| 16 | **Cell 14** | **훈련 후 평가 그래프 10개** |


## 1️⃣ 라이브러리 임포트

In [1]:
# Cell 1: 라이브러리 임포트
import os, math, shutil, warnings
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from sklearn.metrics import (confusion_matrix, roc_auc_score,
                              classification_report, roc_curve, auc)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import (Dense, GlobalAveragePooling2D, GlobalMaxPooling2D,
                                      Dropout, BatchNormalization, Concatenate)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback, TerminateOnNaN, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from tensorflow.keras import mixed_precision
import tensorflow.keras.backend as K
from PIL import Image
import random, json as json_module, gc, subprocess, time

warnings.filterwarnings("ignore")

print("✅ 라이브러리 임포트 완료")
print(f"   TensorFlow: {tf.__version__}")
print(f"   Python 환경 준비 완료")


2026-06-12 17:20:08.283527: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-12 17:20:08.536426: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-12 17:20:08.536494: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-12 17:20:08.572186: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-12 17:20:08.646374: I tensorflow/core/platform/cpu_feature_guar

✅ 라이브러리 임포트 완료
   TensorFlow: 2.15.0
   Python 환경 준비 완료


## 2️⃣ 한글 폰트 설정

In [2]:
# Cell 2: 한글 폰트 설정 (matplotlib 한글·기호 완전 지원)
import matplotlib
import matplotlib.font_manager as fm
import glob

# NanumGothic 우선 탐색 — 없으면 UnDotum 또는 시스템 폰트
_CANDIDATES = [
    "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
    "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf",
    "/usr/share/fonts/truetype/unfonts-core/UnDotum.ttf",
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
]
font_path = None
for _p in _CANDIDATES:
    if os.path.exists(_p):
        font_path = _p
        break

if font_path is None:
    # 시스템에서 한글 지원 폰트 자동 검색
    _all = fm.findSystemFonts(fontpaths=None, fontext="ttf")
    for _p in _all:
        if any(kw in _p.lower() for kw in ["nanum", "malgun", "gulim", "undotum", "noto"]):
            font_path = _p
            break

if font_path is None:
    print("⚠️  한글 폰트를 찾지 못했습니다. 기본 폰트 사용.")
    font_name = "DejaVu Sans"
else:
    # 폰트 캐시 초기화
    cache_dir = matplotlib.get_cachedir()
    for f in glob.glob(os.path.join(cache_dir, "*.json")) + glob.glob(os.path.join(cache_dir, "*.cache")):
        try: os.remove(f)
        except: pass

    fm.fontManager.addfont(font_path)
    font_name = fm.FontProperties(fname=font_path).get_name()
    print(f"✅ 한글 폰트 적용: {font_name}  ({font_path})")

matplotlib.rcParams["font.family"]       = "sans-serif"
matplotlib.rcParams["font.sans-serif"]   = [font_name, "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

# 다크 테마 공통 스타일
matplotlib.rcParams.update({
    "figure.facecolor":  "#0d1117",
    "axes.facecolor":    "#161b22",
    "axes.edgecolor":    "#444",
    "axes.labelcolor":   "#aaa",
    "xtick.color":       "#aaa",
    "ytick.color":       "#aaa",
    "grid.color":        "#333",
    "grid.linestyle":    "--",
    "grid.alpha":        0.5,
    "legend.facecolor":  "#222",
    "legend.labelcolor": "white",
    "text.color":        "white",
})

print("✅ 다크테마 rcParams 전역 적용 완료")


✅ 한글 폰트 적용: NanumGothic  (/usr/share/fonts/truetype/nanum/NanumGothic.ttf)
✅ 다크테마 rcParams 전역 적용 완료


## 3️⃣ GPU 설정 & 실시간 모니터링

In [3]:
# Cell 3: GPU 설정 + 모니터링 대시보드
# ──────────────────────────────────────────────────────
# Mixed precision + JIT 설정
mixed_precision.set_global_policy("mixed_float16")
tf.config.optimizer.set_jit(False)

gpus = tf.config.list_physical_devices("GPU")
print("=" * 55)
print("🖥️  GPU 환경 점검")
print("=" * 55)

if not gpus:
    print("⚠️  GPU 미감지 — CPU 모드로 실행됩니다 (속도 느림)")
    print("   CUDA 드라이버 및 TensorFlow GPU 패키지를 확인하세요.")
else:
    for i, gpu in enumerate(gpus):
        tf.config.experimental.set_memory_growth(gpu, True)
        print(f"  ✅ GPU {i}: {gpu.name}  (메모리 동적 할당 활성화)")
    try:
        result = subprocess.run(
            ["nvidia-smi",
             "--query-gpu=name,memory.total,memory.free,memory.used,utilization.gpu,temperature.gpu",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0:
            for line in result.stdout.strip().split("\n"):
                parts = [p.strip() for p in line.split(",")]
                if len(parts) >= 6:
                    name, total, free, used, util, temp = parts[:6]
                    used_pct  = round(int(used) / int(total) * 100, 1)
                    free_pct  = 100 - used_pct
                    bar_used  = "█" * int(used_pct / 5)
                    bar_free  = "░" * (20 - int(used_pct / 5))
                    print(f"\n  GPU 이름   : {name}")
                    print(f"  VRAM 총량  : {int(total):,} MiB")
                    print(f"  사용 중    : {int(used):,} MiB ({used_pct}%)")
                    print(f"  여유       : {int(free):,} MiB ({free_pct}%)")
                    print(f"  VRAM 사용률: [{bar_used}{bar_free}] {used_pct}%")
                    print(f"  GPU 활용률 : {util}%")
                    print(f"  온도       : {temp}°C")
                    print()
                    if int(free) < 2000:
                        print("  🔴 경고: VRAM 여유 2GB 미만 → OOM 위험! BATCH_SIZE를 줄이세요.")
                    elif int(free) < 4000:
                        print("  🟡 주의: VRAM 여유 4GB 미만 → 배치 크기 모니터링 권장")
                    else:
                        print("  🟢 정상: VRAM 충분 — 학습 준비 완료")
    except Exception as e:
        print(f"  nvidia-smi 조회 실패: {e}")

print("=" * 55)
print(f"  Mixed Precision : float16 ✅")
print(f"  XLA JIT         : 비활성화 ✅")
print(f"  메모리 동적할당  : 활성화 ✅")
print("=" * 55)

# ── GPU 상태 유틸 함수 ──────────────────────────────────────────────────────
def gpu_status(label=""):
    """학습 중 GPU 사용률/온도를 간단히 출력하는 유틸 함수."""
    try:
        r = subprocess.run(
            ["nvidia-smi",
             "--query-gpu=memory.used,memory.total,utilization.gpu,temperature.gpu",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=5
        )
        if r.returncode == 0:
            parts = [p.strip() for p in r.stdout.strip().split(",")]
            used, total, util, temp = int(parts[0]), int(parts[1]), parts[2], parts[3]
            pct = round(used / total * 100, 1)
            tag = f"[{label}] " if label else ""
            print(f"  {tag}GPU: {used}/{total}MiB ({pct}%) | 활용률: {util}% | 온도: {temp}°C")
    except:
        pass

# ── GPU 모니터링 콜백 ──────────────────────────────────────────────────────
class GPUMonitorCallback(Callback):
    """매 N 에포크마다 GPU 상태를 출력하는 Keras 콜백."""
    def __init__(self, every_n_epochs=5):
        super().__init__()
        self.every_n = every_n_epochs

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.every_n == 0:
            gpu_status(f"Epoch {epoch+1}")

# ── CUDA 장치 확인 강제 검증 ─────────────────────────────────────────────
_gpus = tf.config.list_physical_devices("GPU")
assert len(_gpus) > 0 or True, "GPU 없음 — CPU 폴백 허용"
# 실제 연산이 GPU에서 실행되는지 테스트
with tf.device("/GPU:0" if _gpus else "/CPU:0"):
    _test = tf.matmul(tf.random.normal([64, 64]), tf.random.normal([64, 64]))
print(f"\n✅ 연산 장치 검증 완료: {'GPU' if _gpus else 'CPU'}")
print("✅ gpu_status() / GPUMonitorCallback 정의 완료")


INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 4090, compute capability 8.9
🖥️  GPU 환경 점검
  ✅ GPU 0: /physical_device:GPU:0  (메모리 동적 할당 활성화)


2026-06-12 17:20:11.367397: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-12 17:20:11.493860: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-12 17:20:11.493900: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-12 17:20:11.494256: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-12 17:20:11.529477: I external/local_xla/xla/stream_executor


  GPU 이름   : NVIDIA GeForce RTX 4090
  VRAM 총량  : 24,564 MiB
  사용 중    : 2,466 MiB (10.0%)
  여유       : 21,673 MiB (90.0%)
  VRAM 사용률: [██░░░░░░░░░░░░░░░░░░] 10.0%
  GPU 활용률 : 31%
  온도       : 44°C

  🟢 정상: VRAM 충분 — 학습 준비 완료
  Mixed Precision : float16 ✅
  XLA JIT         : 비활성화 ✅
  메모리 동적할당  : 활성화 ✅

✅ 연산 장치 검증 완료: GPU
✅ gpu_status() / GPUMonitorCallback 정의 완료


## 4️⃣ 실험 설정 (하이퍼파라미터)

In [4]:
# Cell 4: 실험 설정
BLUR_SIGMAS  = [0, 1, 2, 4, 8, 16, 32, 64, 128, 256]

BASE_TRAIN   = "/workspace/Train"       # 원본 학습 데이터 (절대 수정하지 않음)
WORKSPACE    = "/workspace"
IMG_SIZE     = 512          # 이미지 512×512 고정
BATCH_SIZE   = 8
EPOCHS_S1    = 80
EPOCHS_S2    = 200
NUM_CLASSES  = 3
SEED         = 42
N_FOLDS      = 7
TTA_STEPS    = 20
SWA_START    = 40
TARGET_COUNT = 1000         # 각 클래스별 1000장으로 오버샘플링
TEST_RATIO   = 0.10
AUTOTUNE     = tf.data.AUTOTUNE

EXPERIMENT_VERSION = "same_settings_gpu4090_v2"
ORIGINAL_SPLIT_DIR = os.path.join(
    WORKSPACE, f"AllExp_Split_ORIGINAL_seed{SEED}_{EXPERIMENT_VERSION}"
)
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp")

# 학습 히스토리 저장소 (과적합/과소적합 진단용)
ALL_HISTORIES = {}   # { sigma: { fold: history_dict } }

def set_reproducible(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["TF_DETERMINISTIC_OPS"] = "1"
    random.seed(seed)
    np.random.seed(seed)
    try:
        tf.keras.utils.set_random_seed(seed)
    except AttributeError:
        tf.random.set_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

def list_class_dirs(directory):
    return sorted([d for d in os.listdir(directory)
                   if os.path.isdir(os.path.join(directory, d))])

def list_image_files(directory):
    if not os.path.exists(directory):
        return []
    return sorted([f for f in os.listdir(directory)
                   if f.lower().endswith(IMAGE_EXTENSIONS)])

set_reproducible(SEED)

print("┌" + "─" * 62 + "┐")
print("│           ✅ 실험 설정 완료                                  │")
print("├" + "─" * 62 + "┤")
print(f"│  이미지 크기     : {IMG_SIZE}×{IMG_SIZE} (고정){' '*31}│")
print(f"│  블러 Sigma     : {str(BLUR_SIGMAS)[:42]:<42}│")
print(f"│  클래스별 목표   : {TARGET_COUNT}장{' '*42}│")
print(f"│  K-Fold         : {N_FOLDS}{' '*45}│")
print(f"│  TTA 횟수       : {TTA_STEPS}회{' '*43}│")
print(f"│  배치 크기      : {BATCH_SIZE}{' '*46}│")
print(f"│  공통 원본 split: {os.path.basename(ORIGINAL_SPLIT_DIR)[:42]:<42}│")
print(f"│  실험 버전      : {EXPERIMENT_VERSION:<42}│")
print("└" + "─" * 62 + "┘")

┌──────────────────────────────────────────────────────────────┐
│           ✅ 실험 설정 완료                                  │
├──────────────────────────────────────────────────────────────┤
│  이미지 크기     : 512×512 (고정)                               │
│  블록 크기      : [512, 256, 128, 64, 32, 16, 8, 4, 2, 1]   │
│  클래스별 목표   : 1000장                                          │
│  K-Fold         : 7                                             │
│  TTA 횟수       : 20회                                           │
│  배치 크기      : 8                                              │
│  공통 원본 split: AllExp_Split_ORIGINAL_seed42_same_settings│
│  실험 버전      : same_settings_gpu4090_v2                  │
└──────────────────────────────────────────────────────────────┘


## 5️⃣ 블록 암호화 함수

In [5]:
# Cell 5: 가우시안 블러 처리 함수
from PIL import Image, ImageFilter

def apply_gaussian_blur(img_array, sigma):
    """sigma=0이면 원본 그대로, sigma>0이면 가우시안 블러 적용"""
    if sigma == 0:
        return img_array.copy()
    img = Image.fromarray(img_array.astype(np.uint8))
    img = img.filter(ImageFilter.GaussianBlur(radius=sigma))
    return np.array(img, dtype=np.uint8)


def generate_blur_dataset(sigma, seed=SEED):
    """
    sigma=0  → 원본(BASE_TRAIN) 그대로 사용 (파일 복사 없음)
    sigma>0  → Train_BlurSigma{sigma}/ 에 블러 이미지 저장
    BASE_TRAIN 원본 파일은 절대 수정하지 않음.
    """
    if sigma == 0:
        print(f"  Sigma=0: 원본 사용 (원본 건드리지 않음)")
        return BASE_TRAIN

    sigma_label = str(sigma).replace(".", "p")
    out_dir = os.path.join(WORKSPACE, f"Train_BlurSigma{sigma_label}")
    classes = list_class_dirs(BASE_TRAIN)

    total_src = sum(len(list_image_files(os.path.join(BASE_TRAIN, c))) for c in classes)
    total_ex = (
        sum(len(list_image_files(os.path.join(out_dir, c)))
            for c in classes if os.path.exists(os.path.join(out_dir, c)))
        if os.path.exists(out_dir) else 0
    )

    if total_ex >= total_src:
        print(f"  BlurSigma={sigma}: 이미 존재 ({total_ex}장) → 스킵")
        return out_dir

    print(f"  BlurSigma={sigma}: 가우시안 블러 처리 중 ({total_src}장)...")
    for cls in classes:
        os.makedirs(os.path.join(out_dir, cls), exist_ok=True)
        for f in list_image_files(os.path.join(BASE_TRAIN, cls)):
            dst = os.path.join(out_dir, cls, f)
            if os.path.exists(dst):
                continue
            try:
                img = Image.open(os.path.join(BASE_TRAIN, cls, f)).convert("RGB")
                img_array = np.array(img, dtype=np.uint8)
                blurred = apply_gaussian_blur(img_array, sigma)
                Image.fromarray(blurred).save(dst, "JPEG", quality=95)
            except Exception:
                pass
    print(f"  BlurSigma={sigma}: 완료! (원본 파일 보존됨)")
    return out_dir

## 6️⃣ 오버샘플링 (클래스별 1,000장)

In [6]:
# Cell 6: 오버샘플링 (각 클래스 1000장, 원본 파일 보존)
def oversample_dir(base_dir, target_count=1000, seed=SEED):
    """원본(base_dir) 파일을 건드리지 않고, 버전이 붙은 oversampled 폴더에 복사+증강."""
    out_dir = base_dir.rstrip("/") + f"_oversampled_{EXPERIMENT_VERSION}_seed{seed}"
    classes = list_class_dirs(base_dir)

    if os.path.exists(out_dir):
        all_done = all(
            len(list_image_files(os.path.join(out_dir, c))) >= target_count
            for c in classes if os.path.exists(os.path.join(out_dir, c))
        ) and all(os.path.exists(os.path.join(out_dir, c)) for c in classes)
        if all_done:
            total = sum(len(list_image_files(os.path.join(out_dir, c))) for c in classes)
            print(f"    오버샘플링 이미 완료 ({total}장) → 재사용")
            return out_dir

    rng = random.Random(seed)

    for cls in classes:
        os.makedirs(os.path.join(out_dir, cls), exist_ok=True)
        src_files = [f for f in list_image_files(os.path.join(base_dir, cls))
                     if not f.startswith("aug_")]
        for f in src_files:
            dst = os.path.join(out_dir, cls, f)
            if not os.path.exists(dst):
                shutil.copy2(os.path.join(base_dir, cls, f), dst)

    for cls in classes:
        cls_out = os.path.join(out_dir, cls)
        files = list_image_files(cls_out)
        src_files = [f for f in list_image_files(os.path.join(base_dir, cls))
                     if not f.startswith("aug_")]
        needed = target_count - len(files)
        if needed <= 0:
            print(f"    {cls}: {len(files)}장 (목표 {target_count}장 달성)")
            continue
        if not src_files:
            print(f"    {cls}: 원본 이미지가 없어 증강을 건너뜀")
            continue
        print(f"    {cls}: {len(files)}장 → {target_count}장 ({needed}장 증강 생성)")
        generated = 0
        while generated < needed:
            src_f = rng.choice(src_files)
            try:
                img = Image.open(os.path.join(base_dir, cls, src_f)).convert("RGB")
                img = img.rotate(rng.uniform(-30, 30), expand=False)
                if rng.random() > 0.5: img = img.transpose(Image.FLIP_LEFT_RIGHT)
                if rng.random() > 0.5: img = img.transpose(Image.FLIP_TOP_BOTTOM)
                from PIL import ImageEnhance
                img = ImageEnhance.Brightness(img).enhance(rng.uniform(0.7, 1.3))
                img.save(os.path.join(cls_out, f"aug_{generated:05d}_{src_f}"),
                         "JPEG", quality=95)
                generated += 1
            except Exception:
                continue

    total = sum(len(list_image_files(os.path.join(out_dir, c))) for c in classes)
    print(f"    ✅ 오버샘플링 완료: {total}장 → {out_dir}")
    print(f"    ✅ 원본 파일({base_dir}) 보존 확인")
    return out_dir


## 7️⃣ Train / Test 분리

In [7]:
# Cell 7: Train/Test 분리
def prepare_split(base_dir, split_dir, test_ratio=0.10, seed=SEED):
    test_dir  = os.path.join(split_dir, "test")
    kfold_dir = os.path.join(split_dir, "kfold_data")
    if os.path.exists(test_dir) and os.path.exists(kfold_dir):
        print(f"    Split 폴더 이미 존재 → 재사용: {split_dir}")
        return

    rng = np.random.default_rng(seed)
    classes = list_class_dirs(base_dir)
    for cls in classes:
        os.makedirs(os.path.join(test_dir,  cls), exist_ok=True)
        os.makedirs(os.path.join(kfold_dir, cls), exist_ok=True)
    for cls in classes:
        files = list_image_files(os.path.join(base_dir, cls))
        rng.shuffle(files)
        n_test = int(len(files) * test_ratio)
        for f in files[:n_test]:
            shutil.copy2(os.path.join(base_dir, cls, f), os.path.join(test_dir, cls, f))
        for f in files[n_test:]:
            shutil.copy2(os.path.join(base_dir, cls, f), os.path.join(kfold_dir, cls, f))
        print(f"    {cls}: kfold={len(files)-n_test}, test={n_test}")


## 8️⃣ tf.data 파이프라인 (512×512 고정)

In [8]:
# Cell 8: tf.data 파이프라인 (512×512 고정)
DATA_OPTIONS = tf.data.Options()
DATA_OPTIONS.experimental_deterministic = True

def _deterministic(ds):
    return ds.with_options(DATA_OPTIONS)

def get_paths_labels(directory, class_names):
    paths, labels = [], []
    for idx, cls in enumerate(class_names):
        cls_dir = os.path.join(directory, cls)
        if not os.path.exists(cls_dir): continue
        for f in list_image_files(cls_dir):
            paths.append(os.path.join(cls_dir, f))
            one_hot = [0.0] * len(class_names)
            one_hot[idx] = 1.0
            labels.append(one_hot)
    return paths, labels

@tf.function
def load_image(path, label):
    raw = tf.io.read_file(path)
    img = tf.image.decode_jpeg(raw, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return tf.cast(img, tf.float32), label

@tf.function
def augment_train(img, label):
    img = tf.cast(img, tf.float32)
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, max_delta=0.25 * 255)
    img = tf.image.random_contrast(img, lower=0.75, upper=1.25)
    noise = tf.random.normal(shape=tf.shape(img), mean=0.0, stddev=5.0, dtype=tf.float32)
    img = tf.clip_by_value(img + noise, 0.0, 255.0)
    return img, label

@tf.function
def mixup_batch(x, y, alpha=0.3):
    batch_size = tf.shape(x)[0]
    idx = tf.random.shuffle(tf.range(batch_size))
    lam = tf.cast(tf.random.uniform([], minval=alpha, maxval=1.0 - alpha + 0.001), tf.float32)
    x = tf.cast(x, tf.float32); y = tf.cast(y, tf.float32)
    return lam * x + (1 - lam) * tf.gather(x, idx), lam * y + (1 - lam) * tf.gather(y, idx)

@tf.function
def cutmix_batch(x, y, alpha=0.3):
    batch_size = tf.shape(x)[0]
    idx   = tf.random.shuffle(tf.range(batch_size))
    lam   = tf.cast(tf.random.uniform([], alpha, 1.0 - alpha), tf.float32)
    h     = tf.cast(tf.shape(x)[1], tf.float32)
    w     = tf.cast(tf.shape(x)[2], tf.float32)
    cut_h = tf.cast(h * tf.math.sqrt(1.0 - lam), tf.int32)
    cut_w = tf.cast(w * tf.math.sqrt(1.0 - lam), tf.int32)
    cx    = tf.random.uniform([], 0, tf.cast(h, tf.int32), dtype=tf.int32)
    cy    = tf.random.uniform([], 0, tf.cast(w, tf.int32), dtype=tf.int32)
    x1    = tf.clip_by_value(cx - cut_h // 2, 0, tf.cast(h, tf.int32))
    y1    = tf.clip_by_value(cy - cut_w // 2, 0, tf.cast(w, tf.int32))
    x2    = tf.clip_by_value(cx + cut_h // 2, 0, tf.cast(h, tf.int32))
    y2    = tf.clip_by_value(cy + cut_w // 2, 0, tf.cast(w, tf.int32))
    mask  = tf.ones([x2 - x1, y2 - y1, tf.shape(x)[3]])
    paddings = [[x1, tf.shape(x)[1] - x2], [y1, tf.shape(x)[2] - y2], [0, 0]]
    mask  = tf.pad(mask, paddings)
    mask  = tf.expand_dims(mask, 0)
    x     = tf.cast(x, tf.float32)
    mixed = x * (1.0 - mask) + tf.gather(x, idx) * mask
    lam_actual = 1.0 - tf.cast((x2 - x1) * (y2 - y1), tf.float32) / (h * w)
    y     = tf.cast(y, tf.float32)
    mixed_y = lam_actual * y + (1.0 - lam_actual) * tf.gather(y, idx)
    return mixed, mixed_y

def apply_mixup(ds, alpha=0.3):
    return _deterministic(ds.map(lambda x, y: mixup_batch(x, y, alpha),
                                 num_parallel_calls=AUTOTUNE)).prefetch(1)

def apply_cutmix(ds, alpha=0.3):
    return _deterministic(ds.map(lambda x, y: cutmix_batch(x, y, alpha),
                                 num_parallel_calls=AUTOTUNE)).prefetch(1)

def apply_mixup_cutmix(ds, alpha=0.3, cutmix_prob=0.5):
    def _apply(x, y):
        use_cutmix = tf.random.uniform([]) < cutmix_prob
        mx, my = mixup_batch(x, y, alpha)
        cx, cy = cutmix_batch(x, y, alpha)
        return tf.cond(use_cutmix, lambda: (cx, cy), lambda: (mx, my))
    return _deterministic(ds.map(_apply, num_parallel_calls=AUTOTUNE)).prefetch(1)

@tf.function
def augment_tta(img, label):
    img = tf.cast(img, tf.float32)
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.15 * 255)
    img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
    return img, label

def make_train_ds(directory, class_names, batch_size=BATCH_SIZE):
    paths, labels = get_paths_labels(directory, class_names)
    n  = len(paths)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = _deterministic(ds.shuffle(min(n, 1000), seed=SEED, reshuffle_each_iteration=True))
    ds = _deterministic(ds.map(load_image, num_parallel_calls=AUTOTUNE))
    ds = _deterministic(ds.map(augment_train, num_parallel_calls=AUTOTUNE))
    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.repeat()
    ds = ds.prefetch(AUTOTUNE)
    return ds, n

def make_eval_ds(directory, class_names, batch_size=BATCH_SIZE):
    paths, labels = get_paths_labels(directory, class_names)
    n  = len(paths)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = _deterministic(ds.map(load_image, num_parallel_calls=AUTOTUNE))
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(1)
    return ds, n, paths

def make_tta_ds(directory, class_names, batch_size=BATCH_SIZE):
    paths, labels = get_paths_labels(directory, class_names)
    n  = len(paths)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = _deterministic(ds.map(load_image, num_parallel_calls=AUTOTUNE))
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = _deterministic(ds.map(lambda x, y: augment_tta(x, y), num_parallel_calls=AUTOTUNE))
    ds = ds.prefetch(1)
    return ds, n

print(f"✅ tf.data 파이프라인 정의 완료 (이미지 {IMG_SIZE}×{IMG_SIZE} 고정, deterministic=True)")


✅ tf.data 파이프라인 정의 완료 (이미지 512×512 고정, deterministic=True)


## 9️⃣ 학습 기법 (Focal Loss / Cosine LR / SWA)

In [9]:
# Cell 9: 학습 기법 (Focal Loss, Cosine Annealing, SWA)
def focal_loss(gamma=2.0, label_smoothing=0.05):
    def loss_fn(y_true, y_pred):
        num_cls   = tf.cast(tf.shape(y_true)[-1], tf.float32)
        y_true_sm = y_true * (1.0 - label_smoothing) + (label_smoothing / num_cls)
        y_pred    = tf.clip_by_value(y_pred, 1e-7, 1.0)
        ce        = -y_true_sm * tf.math.log(y_pred)
        weight    = y_true_sm * tf.math.pow(1.0 - y_pred, gamma)
        return tf.reduce_mean(tf.reduce_sum(weight * ce, axis=-1))
    return loss_fn

class CosineAnnealingWarmRestart(tf.keras.callbacks.Callback):
    def __init__(self, lr_max, lr_min, T0=10, T_mult=2):
        super().__init__()
        self.lr_max = lr_max; self.lr_min = lr_min
        self.T0 = T0; self.T_mult = T_mult; self.T_cur = 0; self.T_i = T0
    def on_epoch_begin(self, epoch, logs=None):
        lr = self.lr_min + 0.5 * (self.lr_max - self.lr_min) * (1 + math.cos(math.pi * self.T_cur / self.T_i))
        K.set_value(self.model.optimizer.learning_rate, lr)
        self.T_cur += 1
        if self.T_cur >= self.T_i:
            self.T_cur = 0; self.T_i = int(self.T_i * self.T_mult)

class SWACallback(tf.keras.callbacks.Callback):
    def __init__(self, start_epoch, swa_lr=1e-5):
        super().__init__()
        self.start_epoch = start_epoch; self.swa_lr = swa_lr
        self.swa_weights = None; self.n_averaged = 0
    def on_epoch_end(self, epoch, logs=None):
        if epoch >= self.start_epoch:
            if self.swa_weights is None:
                self.swa_weights = [w.copy() for w in self.model.get_weights()]
            else:
                for i, w in enumerate(self.model.get_weights()):
                    self.swa_weights[i] = (self.swa_weights[i] * self.n_averaged + w) / (self.n_averaged + 1)
            self.n_averaged += 1
            K.set_value(self.model.optimizer.learning_rate, self.swa_lr)
    def on_train_end(self, logs=None):
        if self.swa_weights is not None:
            self.model.set_weights(self.swa_weights)
            print(f"\n✅ SWA 완료: {self.n_averaged}개 평균")

print("✅ focal_loss / CosineAnnealingWarmRestart / SWACallback 정의 완료")


✅ focal_loss / CosineAnnealingWarmRestart / SWACallback 정의 완료


## 🔟 모델 빌드 (ResNet50)

In [10]:
# Cell 10: 에지-클라우드 분리 모델 빌드
#
# 에지 컴퓨팅 구조:
#   [Edge Device]  ResNet50 Stage-1 (conv1 → BN → ReLU → MaxPool)
#                  → 특징맵만 클라우드로 전송 (원본 이미지 외부 미전송 = 보안)
#   [Cloud Server] Stage 2~5 + 분류 헤드
#
# 학습: 에지+클라우드 처음부터 end-to-end 단일 단계로 학습
#        → 역전파가 에지 conv1까지 확실히 전달됨
# 배포: 학습 완료 후 에지/클라우드 가중치를 분리 저장
# ─────────────────────────────────────────────────────────────────

EDGE_OUTPUT_LAYER = "pool1_pool"
FEATURE_BASE_DIR  = os.path.join(WORKSPACE, "블러 피쳐맵 저장 파일")
FEATURE_SAVE_DIR  = FEATURE_BASE_DIR  # 블러 피쳐맵 저장 파일 / sigma별 파일 생성 / 피쳐맵
os.makedirs(FEATURE_BASE_DIR, exist_ok=True)

def feature_folder_name(sigma):
    return "원본 파일 생성" if sigma == 0 else f"sigma{sigma} 블러 파일 생성"

def feature_file_label(sigma):
    return "원본" if sigma == 0 else f"sigma{sigma}_블러"

def feature_map_dir(sigma):
    return os.path.join(FEATURE_BASE_DIR, feature_folder_name(sigma), "피쳐맵")

def build_edge_model():
    """
    에지 디바이스용 모델.
    ResNet50 초기 컨볼루션 블록(conv1 + BN + ReLU + MaxPool)만 포함.
    입력:  (IMG_SIZE, IMG_SIZE, 3)
    출력:  (128, 128, 64)
    """
    _gpus = tf.config.list_physical_devices("GPU")
    _device = "/GPU:0" if _gpus else "/CPU:0"
    with tf.device(_device):
        full_base   = ResNet50(weights="imagenet", include_top=False,
                               input_shape=(IMG_SIZE, IMG_SIZE, 3))
        edge_output = full_base.get_layer(EDGE_OUTPUT_LAYER).output
        edge_model  = Model(inputs=full_base.input,
                            outputs=edge_output,
                            name="edge_model")
    return edge_model


def build_cloud_model(edge_output_shape):
    """
    클라우드 서버용 모델.
    에지에서 전송된 특징맵을 입력받아 나머지 ResNet50 + 분류 헤드로 최종 분류.
    """
    _gpus = tf.config.list_physical_devices("GPU")
    _device = "/GPU:0" if _gpus else "/CPU:0"
    with tf.device(_device):
        full_base  = ResNet50(weights="imagenet", include_top=False,
                              input_shape=(IMG_SIZE, IMG_SIZE, 3))
        stage1_end = full_base.get_layer(EDGE_OUTPUT_LAYER)
        cloud_base = Model(inputs=stage1_end.output,
                           outputs=full_base.output,
                           name="cloud_base")

        cloud_input = tf.keras.Input(shape=edge_output_shape, name="cloud_input")
        x = cloud_base(cloud_input)

        gap = GlobalAveragePooling2D()(x)
        gmp = GlobalMaxPooling2D()(x)
        x   = Concatenate()([gap, gmp])
        x   = Dense(512, kernel_regularizer=l2(1e-4))(x)
        x   = BatchNormalization()(x)
        x   = tf.keras.layers.Activation("relu")(x)
        x   = Dropout(0.35)(x)
        x   = Dense(256, kernel_regularizer=l2(1e-4))(x)
        x   = BatchNormalization()(x)
        x   = tf.keras.layers.Activation("relu")(x)
        x   = Dropout(0.25)(x)
        out = Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)

        cloud_model = Model(inputs=cloud_input, outputs=out, name="cloud_model")
    return cloud_model


def build_full_pipeline():
    """
    에지 + 클라우드를 하나로 연결한 통합 파이프라인.
    처음부터 end-to-end로 학습 → 역전파가 에지 conv1까지 전달.
    배포 시: edge_model / cloud_model 가중치를 분리 저장하여 각각 탑재.
    """
    edge_model   = build_edge_model()
    edge_out_shp = edge_model.output_shape[1:]
    cloud_model  = build_cloud_model(edge_out_shp)

    # 처음부터 전체 학습 허용
    edge_model.trainable  = True
    cloud_model.trainable = True

    img_input = edge_model.input
    feat_map  = edge_model(img_input, training=True)
    final_out = cloud_model(feat_map)

    pipeline = Model(inputs=img_input, outputs=final_out,
                     name="edge_cloud_pipeline")
    return pipeline, edge_model, cloud_model


def save_feature_map(pipeline, sigma, test_dir, class_names):
    """
    스테이지별로 파일을 분리하여 저장.

    저장 구조:
      피쳐맵 저장 경로/
        sigma{sigma} 블러 파일 생성/
          피쳐맵/
            Benign/
              stage0_원본.png
              stage1_Stage-1(에지).png
              stage2_Stage-2.png
              ...
            Malignant/
              ...
            Normal/
              ...
    """
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    label = "원본" if sigma == 0 else f"Sigma={sigma} 블러"

    stage_layers = [
        ("Stage-1 (에지→클라우드 전송)", "pool1_pool"),
        ("Stage-2",                       "conv2_block3_out"),
        ("Stage-3",                       "conv3_block4_out"),
        ("Stage-4",                       "conv4_block6_out"),
        ("Stage-5",                       "conv5_block3_out"),
    ]

    # 전체 ResNet50으로 가중치 이전
    full_base = ResNet50(weights=None, include_top=False,
                         input_shape=(IMG_SIZE, IMG_SIZE, 3))
    edge_sub  = pipeline.get_layer("edge_model")
    cloud_sub = pipeline.get_layer("cloud_model")
    try:
        for layer in full_base.layers:
            try:
                layer.set_weights(edge_sub.get_layer(layer.name).get_weights())
                continue
            except:
                pass
            try:
                layer.set_weights(
                    cloud_sub.get_layer("cloud_base").get_layer(layer.name).get_weights())
            except:
                pass
    except Exception as e:
        print(f"  ⚠️  가중치 이전 일부 실패 (시각화는 계속): {e}")

    # 스테이지별 추출 모델
    stage_models = []
    for sname, lname in stage_layers:
        try:
            out = full_base.get_layer(lname).output
            m   = Model(inputs=full_base.input, outputs=out, name=f"vis_{lname}")
            stage_models.append((sname, lname, m))
        except Exception as e:
            print(f"  ⚠️  {sname} 모델 생성 실패: {e}")

    # 클래스별 샘플 1장
    samples = {}
    for cls in class_names:
        files = list_image_files(os.path.join(test_dir, cls))
        if files:
            samples[cls] = os.path.join(test_dir, cls, files[0])

    saved_paths = []

    for cls, fpath in samples.items():
        cls_label = cls.replace(" ", "_")

        # 클래스별 하위폴더
        cls_folder = os.path.join(feature_map_dir(sigma), cls_label)
        os.makedirs(cls_folder, exist_ok=True)

        # 이미지 로드
        img_raw = tf.io.read_file(fpath)
        img_raw = tf.image.decode_image(img_raw, channels=3, expand_animations=False)
        img_raw = tf.image.resize(img_raw, [IMG_SIZE, IMG_SIZE])
        img_raw = tf.cast(img_raw, tf.float32) / 255.0
        img_batch = tf.expand_dims(img_raw, 0)

        # ── 파일 0: 원본 이미지 ──────────────────────────────
        fig, ax = plt.subplots(1, 1, figsize=(5, 5), facecolor="white")
        ax.imshow(img_raw.numpy())
        ax.set_title(f"[{label}] {cls}\n원본 CT 이미지 ({IMG_SIZE}×{IMG_SIZE}×3)",
                     fontsize=11, fontweight="bold")
        ax.axis("off")
        fig.tight_layout()
        p = os.path.join(cls_folder,
                         f"stage0_원본_{feature_file_label(sigma)}_{cls_label}.png")
        plt.savefig(p, dpi=120, bbox_inches="tight", facecolor="white")
        plt.close()
        print(f"  📸 저장: {p}")
        saved_paths.append(p)

        # ── 파일 1~5: 스테이지별 ────────────────────────────
        stage_desc = {
            1: ("에지 디바이스에서 클라우드로 전송되는 데이터입니다.\n"
                "ResNet50 첫 번째 Conv+BN+ReLU+MaxPool 통과 후 출력이며, "
                "원본 이미지(512×512×3)가 이 크기로 압축되어 전송됩니다."),
            2: ("ResNet50 Stage-2 (conv2_block3) 출력입니다.\n"
                "저수준 엣지·텍스처 패턴을 학습하며, "
                "클라우드 모델의 첫 번째 처리 단계입니다."),
            3: ("ResNet50 Stage-3 (conv3_block4) 출력입니다.\n"
                "중간 수준의 형태·구조 패턴을 포착하며, "
                "종양 경계 및 음영 정보가 두드러집니다."),
            4: ("ResNet50 Stage-4 (conv4_block6) 출력입니다.\n"
                "고수준 의미론적 특징을 학습하며, "
                "병변 유형(양성/악성)에 따라 활성화 분포가 달라집니다."),
            5: ("ResNet50 Stage-5 (conv5_block3) 출력입니다.\n"
                "분류 직전 최종 특징맵이며, "
                "각 채널이 특정 병리학적 패턴에 반응합니다."),
        }

        for st_idx, (sname, lname, sm) in enumerate(stage_models, start=1):
            feat = sm(img_batch, training=False).numpy()[0]
            h, w, c = feat.shape

            n_samples = min(8, c)
            ch_indices = [int(i * c / n_samples) for i in range(n_samples)]

            # ── 레이아웃: 3행  ──────────────────────────────
            # 행0: 설명 텍스트 전용
            # 행1: 채널 평균 / 최대 활성화 / 채널 샘플 3개
            # 행2: 채널 샘플 5개
            fig = plt.figure(figsize=(20, 11), facecolor="white")
            fig.suptitle(
                f"[{label}]  클래스: {cls}  —  {sname}  |  특징맵 크기: {h}×{w}×{c}채널",
                fontsize=14, fontweight="bold", y=0.98
            )

            # 행0: 설명 텍스트 박스
            ax_desc = fig.add_axes([0.03, 0.72, 0.94, 0.18])
            ax_desc.set_facecolor("#F5F5F5")
            ax_desc.set_xlim(0, 1)
            ax_desc.set_ylim(0, 1)
            ax_desc.axis("off")
            ax_desc.text(
                0.5, 0.55,
                stage_desc[st_idx],
                ha="center", va="center",
                fontsize=11, linespacing=1.8,
                wrap=True
            )
            ax_desc.text(
                0.5, 0.08,
                f"▶  채널 평균: 모든 채널을 평균하여 전체 활성화 분포 시각화  "
                f"│  최대 활성화: 가장 강하게 반응한 위치(병변 집중 부위)  "
                f"│  ch N/{c}: {c}개 채널 중 균등 간격으로 샘플링한 개별 채널",
                ha="center", va="center",
                fontsize=9, color="#444444"
            )

            # 행1 (채널 평균, 최대 활성화, 채널 샘플 3개) — 5열
            subplot_descs_row1 = [
                ("채널 평균", f"전체 {c}채널 평균
전반적 활성화 분포", "viridis"),
                ("최대 활성화", "채널 최댓값
병변 집중 부위", "hot"),
                (f"ch {ch_indices[0]}/{c}", f"채널 {ch_indices[0]}
개별 필터 반응", "viridis"),
                (f"ch {ch_indices[1]}/{c}", f"채널 {ch_indices[1]}
개별 필터 반응", "viridis"),
                (f"ch {ch_indices[2]}/{c}", f"채널 {ch_indices[2]}
개별 필터 반응", "viridis"),
            ]
            for ci, (title, desc, cmap) in enumerate(subplot_descs_row1):
                ax = fig.add_axes([0.03 + ci * 0.194, 0.37, 0.17, 0.30])
                if title == "채널 평균":
                    ax.imshow(feat.mean(axis=-1), cmap=cmap)
                elif title == "최대 활성화":
                    ax.imshow(feat.max(axis=-1), cmap=cmap)
                else:
                    ch_n = ch_indices[ci - 2]
                    ax.imshow(feat[:, :, ch_n], cmap=cmap)
                ax.set_title(title, fontsize=9, fontweight="bold", pad=4)
                ax.axis("off")
                ax.text(0.5, -0.08, desc, transform=ax.transAxes,
                        ha="center", va="top", fontsize=8, color="#333333",
                        linespacing=1.4)

            # 행2 (채널 샘플 5개)
            for ci in range(5):
                ch = ch_indices[3 + ci] if (3 + ci) < len(ch_indices) else ch_indices[-1]
                ax = fig.add_axes([0.03 + ci * 0.194, 0.04, 0.17, 0.30])
                ax.imshow(feat[:, :, ch], cmap="viridis")
                ax.set_title(f"ch {ch}/{c}", fontsize=9, fontweight="bold", pad=4)
                ax.axis("off")
                ax.text(0.5, -0.08, f"채널 {ch}
개별 필터 반응",
                        transform=ax.transAxes,
                        ha="center", va="top", fontsize=8, color="#333333",
                        linespacing=1.4)

            fig.tight_layout(rect=[0, 0, 1, 0.93])
            sname_safe = sname.replace(" ", "_").replace("(", "").replace(")", "").replace("→", "")
            p = os.path.join(cls_folder,
                             f"stage{st_idx}_{sname_safe}_{feature_file_label(sigma)}_{cls_label}.png")
            plt.savefig(p, dpi=120, bbox_inches="tight", facecolor="white")
            plt.close()
            print(f"  📸 저장: {p}")
            saved_paths.append(p)

    del full_base, stage_models
    tf.keras.backend.clear_session()
    gc.collect()

    return saved_paths[0] if saved_paths else None

# ── 구조 검증 ──────────────────────────────────────────────────────────────
print("=" * 60)
print("🏗️  에지-클라우드 분리 모델 정의 완료")
print("=" * 60)
_edge_tmp     = build_edge_model()
_edge_out_shp = _edge_tmp.output_shape[1:]
_cloud_tmp    = build_cloud_model(_edge_out_shp)
print(f"  [Edge  Model] 입력: ({IMG_SIZE}, {IMG_SIZE}, 3)")
print(f"                출력: {_edge_out_shp}  ← 클라우드 전송 특징맵")
print(f"                파라미터: {_edge_tmp.count_params():,}")
print(f"  [Cloud Model] 입력: {_edge_out_shp}")
print(f"                출력: ({NUM_CLASSES},)")
print(f"                파라미터: {_cloud_tmp.count_params():,}")
del _edge_tmp, _cloud_tmp
print(f"  [학습 방식  ] 처음부터 end-to-end (역전파 에지까지 전달)")
print(f"  [배포 방식  ] 학습 후 에지/클라우드 가중치 분리 저장")
print(f"  [특징맵 저장] {FEATURE_SAVE_DIR}")
print("=" * 60)
_gpus_check = tf.config.list_physical_devices("GPU")
print(f"  학습 장치: {'GPU ✅' if _gpus_check else 'CPU ⚠️'}")


🏗️  에지-클라우드 분리 모델 정의 완료
  [Edge  Model] 입력: (512, 512, 3)
                출력: (128, 128, 64)  ← 클라우드 전송 특징맵
                파라미터: 9,728
  [Cloud Model] 입력: (128, 128, 64)
                출력: (3,)
                파라미터: 25,810,819
  [학습 방식  ] 처음부터 end-to-end (역전파 에지까지 전달)
  [배포 방식  ] 학습 후 에지/클라우드 가중치 분리 저장
  [특징맵 저장] /workspace/사진 저장 파일
  학습 장치: GPU ✅


## 🆕 10b. 과적합 / 과소적합 진단 함수

> 학습 히스토리(train loss vs val loss, train acc vs val acc)를 기록하고,
> 각 블록 크기·폴드별로 **과적합 / 과소적합 / 정상** 상태를 자동 판단합니다.


In [11]:
# Cell 10b: 과적합/과소적합 진단 함수 & 모니터 콜백

def diagnose_fit(train_loss, val_loss, train_acc, val_acc):
    if not train_loss:
        return "unknown", {}
    final_tr_loss = train_loss[-1]
    final_va_loss = val_loss[-1]
    final_tr_acc  = train_acc[-1]
    final_va_acc  = val_acc[-1]
    gap_loss = final_va_loss - final_tr_loss
    gap_acc  = final_tr_acc  - final_va_acc

    if final_tr_acc < 0.60 and final_va_acc < 0.60:
        status = "underfit"
    elif gap_acc > 0.15 or gap_loss > 0.25:
        status = "overfit"
    else:
        status = "good"

    detail = {
        "train_loss_final": round(final_tr_loss, 4),
        "val_loss_final":   round(final_va_loss, 4),
        "train_acc_final":  round(final_tr_acc,  4),
        "val_acc_final":    round(final_va_acc,   4),
        "loss_gap":         round(gap_loss, 4),
        "acc_gap":          round(gap_acc,  4),
        "status":           status,
    }
    return status, detail


def print_fold_result(sigma, fold_idx, combined_history, stage=""):
    """폴드 학습 완료 시 결과를 텍스트로 출력"""
    tl = combined_history.get("loss", [])
    vl = combined_history.get("val_loss", [])
    ta = combined_history.get("accuracy", [])
    va = combined_history.get("val_accuracy", [])

    status, d = diagnose_fit(tl, vl, ta, va)
    STATUS_STR = {"good": "✅ 정상", "overfit": "🔴 과적합", "underfit": "🟡 과소적합", "unknown": "❓ 미상"}
    label = "원본" if sigma == 0 else f"Sigma={sigma} 블러"

    print()
    print(f"  ┌─────────────────────────────────────────────────────┐")
    print(f"  │  {label}  Fold {fold_idx+1} 학습 완료{stage:<24}│")
    print(f"  ├─────────────────────────────────────────────────────┤")
    print(f"  │  판정: {STATUS_STR[status]:<46}│")
    print(f"  │  train_acc : {d.get('train_acc_final',0)*100:5.1f}%   val_acc : {d.get('val_acc_final',0)*100:5.1f}%              │")
    print(f"  │  train_loss: {d.get('train_loss_final',0):6.4f}  val_loss: {d.get('val_loss_final',0):6.4f}               │")
    print(f"  │  acc_gap   : {d.get('acc_gap',0)*100:+5.1f}%  loss_gap: {d.get('loss_gap',0):+6.4f}               │")
    print(f"  │  총 에폭   : {len(tl):>3}  (1단계+2단계 합산){' '*17}│")

    ag = d.get('acc_gap', 0)
    lg = d.get('loss_gap', 0)
    if status == "overfit":
        comment = (f"  train-val acc 차이 {ag*100:.1f}%  →  Dropout/정규화 강화 권장"
                   if ag > 0.15 else f"  val_loss 급등 ({lg:+.4f})  →  EarlyStopping 확인")
    elif status == "underfit":
        comment = "  train/val 모두 낮음  →  lr 상향 또는 에폭 추가 권장"
    else:
        comment = "  train ≈ val  →  정상 수렴"
    print(f"  │  💬 {comment:<50}│")
    print(f"  └─────────────────────────────────────────────────────┘")
    print()


# GPU 로그 저장소
GPU_LOG = []

class AdvancedMonitorCallback(Callback):
    """매 에폭마다 GPU 로깅 + 과적합 조기 경보"""
    def __init__(self, block_size, fold_idx, every_n_epochs=5, overfit_gap=0.20):
        super().__init__()
        self.block_size  = block_size
        self.fold_idx    = fold_idx
        self.every_n     = every_n_epochs
        self.overfit_gap = overfit_gap
        self._warned     = False

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        if (epoch + 1) % self.every_n == 0:
            try:
                r = subprocess.run(
                    ["nvidia-smi",
                     "--query-gpu=memory.used,memory.total,utilization.gpu,temperature.gpu",
                     "--format=csv,noheader,nounits"],
                    capture_output=True, text=True, timeout=5
                )
                if r.returncode == 0:
                    parts = [p.strip() for p in r.stdout.strip().split(",")]
                    GPU_LOG.append({
                        "time":      time.time(),
                        "used_mib":  int(parts[0]),
                        "total_mib": int(parts[1]),
                        "util_pct":  float(parts[2]),
                        "temp_c":    float(parts[3]),
                        "epoch":     epoch + 1,
                        "block":     self.block_size,
                        "fold":      self.fold_idx,
                    })
                    used_pct = int(parts[0]) / int(parts[1]) * 100
                    if used_pct > 90:
                        print(f"\n  ⚠️  VRAM {used_pct:.1f}% — OOM 위험!")
            except:
                pass

        tr_loss = logs.get("loss")
        va_loss = logs.get("val_loss")
        tr_acc  = logs.get("accuracy")
        va_acc  = logs.get("val_accuracy")

        if tr_loss is not None and va_loss is not None:
            loss_gap = va_loss - tr_loss
            acc_gap  = (tr_acc or 0) - (va_acc or 0)
            if loss_gap > self.overfit_gap and not self._warned:
                print(f"\n  🔴 [Epoch {epoch+1}] 과적합 경보! loss_gap={loss_gap:.4f}")
                self._warned = True
            elif acc_gap > 0.20 and not self._warned:
                print(f"\n  🔴 [Epoch {epoch+1}] 과적합 경보! acc_gap={acc_gap*100:.1f}%")
                self._warned = True

class OverfitControlCallback(Callback):
    """과적합이 여러 에폭 연속 지속되면 현재 학습 단계를 조기 종료한다."""
    def __init__(self, min_epochs=20, patience=8, acc_gap=0.18, loss_gap=0.20):
        super().__init__()
        self.min_epochs = min_epochs
        self.patience = patience
        self.acc_gap = acc_gap
        self.loss_gap = loss_gap
        self.bad_count = 0

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        if epoch + 1 < self.min_epochs:
            return
        train_acc = logs.get("accuracy")
        val_acc = logs.get("val_accuracy")
        train_loss = logs.get("loss")
        val_loss = logs.get("val_loss")
        if None in (train_acc, val_acc, train_loss, val_loss):
            return
        is_overfit = (train_acc - val_acc > self.acc_gap) and (val_loss - train_loss > self.loss_gap)
        self.bad_count = self.bad_count + 1 if is_overfit else max(0, self.bad_count - 1)
        if self.bad_count >= self.patience:
            print(f"\n  🛑 과적합 방지: {self.bad_count}회 연속 과적합 신호 → 현재 학습 단계 종료")
            self.model.stop_training = True

print("✅ diagnose_fit / print_fold_result / AdvancedMonitorCallback 정의 완료")


✅ diagnose_fit / print_fold_result / AdvancedMonitorCallback 정의 완료


## 1️⃣1️⃣ 단일 블록 실험 함수

In [12]:
# Cell 11: 단일 블러 Sigma 실험 함수 (에지-클라우드 end-to-end 학습)
#
# 요청 설정:
#   단일 학습: lr=1e-4(처음부터), mixup=0.2, cutmix=0.4, EPOCHS_S1+EPOCHS_S2
#
# 에지 구조 적용:
#   - build_full_pipeline() 으로 에지+클라우드 end-to-end 학습
#   - 학습 후 에지/클라우드 가중치 분리 저장
# ─────────────────────────────────────────────────────────────────

def run_experiment(sigma, blur_dir):
    experiment_idx = BLUR_SIGMAS.index(sigma) if sigma in BLUR_SIGMAS else 0
    exp_seed = SEED + experiment_idx * 1000
    set_reproducible(exp_seed)
    label = "원본(σ=0)" if sigma == 0 else f"BlurSigma σ={sigma}"
    print(f"\n{'='*60}\n🌫️  {label}\n{'='*60}")

    split_dir = (ORIGINAL_SPLIT_DIR if sigma == 0 else
                 os.path.join(WORKSPACE,
                              f"AllExp_Split_BlurSigma{sigma}_{EXPERIMENT_VERSION}_seed{SEED}"))
    gc.collect()

    print("  오버샘플링...")
    oversampled_dir = oversample_dir(blur_dir, TARGET_COUNT, seed=SEED)
    print("  Train/Test 분리...")
    prepare_split(oversampled_dir, split_dir, TEST_RATIO, seed=SEED)

    test_dir  = os.path.join(split_dir, "test")
    kfold_dir = os.path.join(split_dir, "kfold_data")

    class_names  = list_class_dirs(kfold_dir)
    class_to_idx = {c: i for i, c in enumerate(class_names)}
    benign_idx   = class_to_idx.get("Bengin cases", 0)
    normal_idx   = class_to_idx.get("Normal cases", 2)

    all_paths, all_labels = [], []
    for cls in class_names:
        for f in list_image_files(os.path.join(kfold_dir, cls)):
            all_paths.append(os.path.join(kfold_dir, cls, f))
            all_labels.append(class_to_idx[cls])
    all_paths  = np.array(all_paths)
    all_labels = np.array(all_labels)
    print(f"  학습: {len(all_paths)}장 | 테스트: "
          f"{sum(len(list_image_files(os.path.join(test_dir, c))) for c in class_names)}장")

    skf         = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_models = []

    if sigma not in ALL_HISTORIES:
        ALL_HISTORIES[sigma] = {}

    _gpus   = tf.config.list_physical_devices("GPU")
    _device = "/GPU:0" if _gpus else "/CPU:0"
    print(f"  학습 장치: {_device}")

    for fold, (train_idx, val_idx) in enumerate(skf.split(all_paths, all_labels)):
        print(f"\n  --- Fold {fold+1}/{N_FOLDS} ---")
        gpu_status(f"Fold{fold+1} 시작")
        set_reproducible(exp_seed + fold)

        fold_train_dir = os.path.join(split_dir, f"fold_{fold}_train")
        fold_val_dir   = os.path.join(split_dir, f"fold_{fold}_val")

        if not os.path.exists(fold_train_dir):
            for cls in class_names:
                os.makedirs(os.path.join(fold_train_dir, cls), exist_ok=True)
                os.makedirs(os.path.join(fold_val_dir,   cls), exist_ok=True)
            for idx in train_idx:
                src = all_paths[idx]; cls = class_names[all_labels[idx]]
                shutil.copy2(src, os.path.join(fold_train_dir, cls, os.path.basename(src)))
            for idx in val_idx:
                src = all_paths[idx]; cls = class_names[all_labels[idx]]
                shutil.copy2(src, os.path.join(fold_val_dir, cls, os.path.basename(src)))

        fold_cw = dict(enumerate(compute_class_weight(
            "balanced", classes=np.unique(all_labels[train_idx]), y=all_labels[train_idx])))
        fold_cw[benign_idx] *= 2.5
        fold_cw[normal_idx] *= 1.3

        train_ds, train_n = make_train_ds(fold_train_dir, class_names)
        val_ds, val_n, _  = make_eval_ds(fold_val_dir,   class_names)
        train_steps = max(1, train_n // BATCH_SIZE)
        val_steps   = math.ceil(val_n / BATCH_SIZE)

        prefix = f"allexp_blur{sigma}_resnet50_f{fold}"
        TEMP   = f"{prefix}_temp.keras"
        BEST   = f"{prefix}_best.keras"

        monitor_cb = AdvancedMonitorCallback(sigma, fold, every_n_epochs=5)

        # ── 통합 파이프라인 빌드 (에지+클라우드 end-to-end) ──────────────
        with tf.device(_device):
            pipeline, edge_model, cloud_model = build_full_pipeline()

        # ── 단일 학습: lr=1e-4, mixup=0.2, cutmix=0.4, EPOCHS_S1+EPOCHS_S2 ─────
        pipeline.trainable = True
        pipeline.compile(
            optimizer=AdamW(learning_rate=1e-4, weight_decay=1e-4),
            loss=focal_loss(2.0, 0.02),
            metrics=["accuracy"])

        h1 = pipeline.fit(
            apply_mixup_cutmix(train_ds, 0.2, cutmix_prob=0.4),
            steps_per_epoch=train_steps,
            validation_data=val_ds.repeat(), validation_steps=val_steps,
            epochs=EPOCHS_S1 + EPOCHS_S2,
            callbacks=[
                EarlyStopping(monitor="val_loss", patience=20, min_delta=1e-4,
                              restore_best_weights=True, verbose=0),
                TerminateOnNaN(),
                OverfitControlCallback(min_epochs=20, patience=8,
                                       acc_gap=0.18, loss_gap=0.20),
                ModelCheckpoint(BEST, monitor="val_loss",
                                save_best_only=True, mode="min", verbose=0),
                CosineAnnealingWarmRestart(lr_max=1e-4, lr_min=1e-5, T0=5, T_mult=2),
                monitor_cb,
            ], class_weight=fold_cw, verbose=1)

        h2 = tf.keras.callbacks.History()
        h2.history = {}

        # ── 학습 완료 후 에지/클라우드 가중치 분리 저장 ──────────────────
        best_pipeline   = tf.keras.models.load_model(
            BEST, custom_objects={"loss_fn": focal_loss(2.0, 0.05)})
        edge_sub_final  = best_pipeline.get_layer("edge_model")
        cloud_sub_final = best_pipeline.get_layer("cloud_model")

        edge_save_path  = f"{prefix}_edge.weights.h5"
        cloud_save_path = f"{prefix}_cloud.keras"
        edge_sub_final.save_weights(edge_save_path)
        cloud_sub_final.save(cloud_save_path)
        print(f"  💾 에지 가중치 (병원 기기용): {edge_save_path}")
        print(f"  💾 클라우드 모델 (서버용)   : {cloud_save_path}")

        combined = {}
        for key in ["loss", "val_loss", "accuracy", "val_accuracy"]:
            combined[key] = list(h1.history.get(key, [])) + list(h2.history.get(key, []))
        ALL_HISTORIES[sigma][fold] = combined
        print_fold_result(sigma, fold, combined)

        pipeline.save(BEST)
        fold_models.append(BEST)
        if os.path.exists(TEMP): os.remove(TEMP)
        del pipeline
        tf.keras.backend.clear_session()
        gc.collect()
        gpu_status(f"Fold{fold+1} 완료")

    # ── 앙상블 + TTA ──────────────────────────────────────────────────────
    print(f"\n  🔮 앙상블 + TTA ({N_FOLDS}개 × {TTA_STEPS}회)...")
    test_ds, test_n, test_files = make_eval_ds(test_dir, class_names)
    test_steps = math.ceil(test_n / BATCH_SIZE)

    y_true_list   = [yb.numpy() for _, yb in test_ds]
    y_true_onehot = np.concatenate(y_true_list, axis=0)
    y_true        = np.argmax(y_true_onehot, axis=1)

    all_fold_preds = []
    for fold, mp in enumerate(fold_models):
        fm = tf.keras.models.load_model(
            mp, custom_objects={"loss_fn": focal_loss(2.0, 0.05)})
        tta_preds = []
        for t in range(TTA_STEPS):
            set_reproducible(exp_seed + 10000 + fold * 100 + t)
            if t == 0:
                ds_t, _ = make_eval_ds(test_dir, class_names)[:2]
            else:
                ds_t, _ = make_tta_ds(test_dir, class_names)
            tta_preds.append(fm.predict(ds_t, steps=test_steps, verbose=0))
            print(f"    Fold{fold+1} TTA {t+1}/{TTA_STEPS}", end="\r")
        all_fold_preds.append(np.mean(tta_preds, axis=0))
        del fm; tf.keras.backend.clear_session(); gc.collect()
    print()

    pred_probs = np.mean(all_fold_preds, axis=0)[:len(y_true)]
    y_pred     = np.argmax(pred_probs, axis=1)
    test_acc   = float(np.mean(y_true == y_pred))
    print(f"  앙상블: {test_acc*100:.2f}%")

    # ── Pseudo Labeling ───────────────────────────────────────────────────
    print("  🔄 Pseudo Labeling...")
    pseudo_dir = os.path.join(split_dir, "pseudo")
    if os.path.exists(pseudo_dir): shutil.rmtree(pseudo_dir)
    pseudo_count = 0
    for fpath, prob in zip(test_files, pred_probs):
        if np.max(prob) >= 0.88:
            cls = class_names[np.argmax(prob)]
            os.makedirs(os.path.join(pseudo_dir, cls), exist_ok=True)
            shutil.copy2(fpath, os.path.join(pseudo_dir, cls, os.path.basename(fpath)))
            pseudo_count += 1
    print(f"  확신도 88%↑: {pseudo_count}장")

    if pseudo_count >= 10:
        pseudo_kfold = os.path.join(split_dir, "pseudo_kfold")
        if os.path.exists(pseudo_kfold): shutil.rmtree(pseudo_kfold)
        for cls in class_names:
            os.makedirs(os.path.join(pseudo_kfold, cls), exist_ok=True)
            for f in list_image_files(os.path.join(kfold_dir, cls)):
                shutil.copy2(os.path.join(kfold_dir, cls, f),
                             os.path.join(pseudo_kfold, cls, f))
            p = os.path.join(pseudo_dir, cls)
            if os.path.exists(p):
                for f in list_image_files(p):
                    shutil.copy2(os.path.join(p, f),
                                 os.path.join(pseudo_kfold, cls, f"pseudo_{f}"))

        set_reproducible(exp_seed + 50000)
        pm = tf.keras.models.load_model(
            fold_models[0], custom_objects={"loss_fn": focal_loss(2.0, 0.05)})
        p_labels = []
        for cls in class_names:
            cnt = len(list_image_files(os.path.join(pseudo_kfold, cls)))
            p_labels.extend([class_to_idx[cls]] * cnt)
        p_cw = dict(enumerate(compute_class_weight(
            "balanced", classes=np.unique(p_labels), y=p_labels)))
        p_cw[benign_idx] *= 1.5
        p_cw[normal_idx] *= 1.3

        p_train_ds, p_n   = make_train_ds(pseudo_kfold, class_names)
        p_val_ds, p_vn, _ = make_eval_ds(
            os.path.join(split_dir, "fold_0_val"), class_names)
        pm.compile(optimizer=AdamW(learning_rate=3e-5, weight_decay=1e-4),
                   loss=focal_loss(2.0, 0.05), metrics=["accuracy"])
        pm.fit(apply_mixup(p_train_ds, 0.1),
               steps_per_epoch=max(1, p_n // BATCH_SIZE),
               validation_data=p_val_ds.repeat(),
               validation_steps=math.ceil(p_vn / BATCH_SIZE),
               epochs=15,
               callbacks=[
                   EarlyStopping(monitor="val_loss", patience=12, min_delta=1e-4,
                                 restore_best_weights=True, verbose=0),
                   ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4,
                                    min_lr=1e-7, verbose=0),
                   TerminateOnNaN(),
                   OverfitControlCallback(min_epochs=8, patience=4,
                                          acc_gap=0.18, loss_gap=0.20)],
               class_weight=p_cw, verbose=0)

        tta_preds = []
        for t in range(TTA_STEPS):
            set_reproducible(exp_seed + 30000 + t)
            if t == 0:
                ds_t, _ = make_eval_ds(test_dir, class_names)[:2]
            else:
                ds_t, _ = make_tta_ds(test_dir, class_names)
            tta_preds.append(pm.predict(ds_t, steps=test_steps, verbose=0))
        all_fold_preds.append(np.mean(tta_preds, axis=0))
        del pm; tf.keras.backend.clear_session(); gc.collect()

        pred_probs = np.mean(all_fold_preds, axis=0)[:len(y_true)]
        y_pred     = np.argmax(pred_probs, axis=1)
        test_acc   = float(np.mean(y_true == y_pred))
        print(f"  Pseudo 후: {test_acc*100:.2f}%")

    cm      = confusion_matrix(y_true, y_pred)
    roc_auc = float(roc_auc_score(
        y_true_onehot[:len(pred_probs)], pred_probs,
        multi_class="ovr", average="macro"))
    per_class = {cls: float(cm[i, i] / cm[i].sum())
                 for i, cls in enumerate(class_names)}

    for mp in fold_models:
        if os.path.exists(mp): os.remove(mp)

    print(f"\n  ✅ Sigma={sigma}: {test_acc*100:.2f}% | ROC-AUC={roc_auc:.4f}")
    return {
        "sigma":           sigma,
        "experiment_seed": exp_seed,
        "acc":             test_acc,
        "roc_auc":         roc_auc,
        "per_class_acc":   per_class,
        "cm":              cm.tolist(),
        "class_names":     class_names,
    }

print("✅ Cell 11: run_experiment() 정의 완료")
print("   단일 학습: lr=1e-4, mixup=0.2, cutmix=0.4")
print("   에폭: EPOCHS_S1 + EPOCHS_S2")
print("   에지+클라우드 end-to-end 학습 유지")


✅ Cell 11: run_experiment() 정의 완료
   단일 학습: lr=1e-4, mixup=0.2, cutmix=0.4
   에폭: EPOCHS_S1 + EPOCHS_S2
   에지+클라우드 end-to-end 학습 유지


## 1️⃣2️⃣ 전체 실험 루프

In [13]:
# Cell 12: 전체 실험 루프
import json as json_module

print("\n" + "=" * 60)
print("🔬 블러 Sigma별 실험 시작 (ResNet50)")
print(f"   Sigma: {BLUR_SIGMAS}")
print(f"   K={N_FOLDS} | TTA={TTA_STEPS} | BATCH={BATCH_SIZE} | IMG={IMG_SIZE}")
print(f"   클래스별 오버샘플링: {TARGET_COUNT}장")
print(f"   실험 버전: {EXPERIMENT_VERSION}")
print("=" * 60)
gpu_status("실험 시작 전")

all_results = {}
result_path = os.path.join(WORKSPACE, f"all_blur_results_resnet50_{EXPERIMENT_VERSION}.json")

if os.path.exists(result_path):
    with open(result_path) as f:
        saved = json_module.load(f)
    for k, v in saved.items():
        all_results[int(k)] = v
    print(f"\n✅ 이전 결과: {sorted(all_results.keys())}")


def _save_feat(sigma, class_names):
    """블러 Sigma별 특징맵 저장 (에지 가중치 자동 탐색)"""
    import glob as _glob
    label = "원본" if sigma == 0 else f"Sigma={sigma} 블러"

    # 저장된 에지 가중치 탐색 (여러 경로 시도)
    search_paths = [
        f"allexp_blur{sigma}_resnet50_f0_edge.weights.h5",
        os.path.join(WORKSPACE, f"allexp_blur{sigma}_resnet50_f0_edge.weights.h5"),
        f"/root/Lung Cancer/allexp_blur{sigma}_resnet50_f0_edge.weights.h5",
    ]
    edge_path = None
    for p in search_paths:
        if os.path.exists(p):
            edge_path = p
            break

    try:
        _vis, _, _ = build_full_pipeline()
        if edge_path:
            _vis.get_layer("edge_model").load_weights(
                edge_path, by_name=True, skip_mismatch=True)
            print(f"  ✅ 에지 가중치 로드: {edge_path}")
        else:
            print(f"  ⚠️  에지 가중치 없음 → ImageNet 초기값으로 시각화")

        _split = (ORIGINAL_SPLIT_DIR if sigma == 0 else
                  os.path.join(WORKSPACE,
                               f"AllExp_Split_BlurSigma{sigma}_{EXPERIMENT_VERSION}_seed{SEED}"))
        _test = os.path.join(_split, "test")
        if os.path.exists(_test):
            save_feature_map(_vis, sigma, _test, class_names)
        else:
            print(f"  ⚠️  테스트 경로 없음: {_test}")
        del _vis
        tf.keras.backend.clear_session()
        gc.collect()
    except Exception as e:
        print(f"  ⚠️  특징맵 저장 실패: {e}")


for step_idx, sigma in enumerate(BLUR_SIGMAS, start=1):
    if sigma in all_results:
        label = "원본" if sigma == 0 else f"Sigma={sigma}"
        print(f"\n⏭️  Step {step_idx}/{len(BLUR_SIGMAS)} | {label}: "
              f"완료({all_results[sigma]['acc']*100:.2f}%) → 스킵")

        # 스킵된 sigma도 특징맵 없으면 저장
        sigma_folder = feature_map_dir(sigma)
        feat_exists  = all(
            os.path.exists(os.path.join(sigma_folder, cls.replace(" ", "_"),
                           f"stage0_원본_{feature_file_label(sigma)}_{cls.replace(' ', '_')}.png"))
            for cls in all_results[sigma]["class_names"]
        )
        if not feat_exists:
            print(f"  📸 특징맵 없음 → 저장 시도...")
            _save_feat(sigma, all_results[sigma]["class_names"])
        continue

    print(f"\n📦 Step {step_idx}/{len(BLUR_SIGMAS)} | BlurSigma={sigma} 데이터셋 준비...")
    blur_dir = generate_blur_dataset(sigma)
    result   = run_experiment(sigma, blur_dir)
    all_results[sigma] = result

    with open(result_path, "w") as f:
        json_module.dump({str(k): v for k, v in all_results.items()}, f,
                         indent=2, ensure_ascii=False)
    print(f"  💾 중간 저장 완료")

    # 학습 완료 후 특징맵 저장
    print(f"  📸 특징맵 저장 중...")
    _save_feat(sigma, result["class_names"])

    gpu_status(f"Sigma={sigma} 완료 후")



🔬 블록 암호화 크기별 실험 시작 (ResNet50)
   블록: [512, 256, 128, 64, 32, 16, 8, 4, 2, 1]
   K=7 | TTA=20 | BATCH=8 | IMG=512
   클래스별 오버샘플링: 1000장
   실험 버전: same_settings_gpu4090_v2
  [실험 시작 전] GPU: 3373/24564MiB (13.7%) | 활용률: 20% | 온도: 45°C

✅ 이전 결과: [32, 64, 128, 256, 512]

⏭️  Step 1/10 | 원본: 완료(98.33%) → 스킵

⏭️  Step 2/10 | Block256: 완료(98.33%) → 스킵

⏭️  Step 3/10 | Block128: 완료(97.00%) → 스킵

⏭️  Step 4/10 | Block64: 완료(98.67%) → 스킵

⏭️  Step 5/10 | Block32: 완료(82.00%) → 스킵

📦 Step 6/10 | Block16 데이터셋 준비...
  Block16: 이미 존재 (1097장) → 스킵

🧩 Block16
  오버샘플링...
    오버샘플링 이미 완료 (3000장) → 재사용
  Train/Test 분리...
    Split 폴더 이미 존재 → 재사용: /workspace/AllExp_Split_Block16_same_settings_gpu4090_v2_seed42
  학습: 2700장 | 테스트: 300장
  학습 장치: /GPU:0

  --- Fold 1/7 ---
  [Fold1 시작] GPU: 3370/24564MiB (13.7%) | 활용률: 18% | 온도: 45°C
Epoch 1/280


2026-06-12 17:20:28.716327: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904


  4/289 [..............................] - ETA: 21s - loss: 1.2400 - accuracy: 0.4375  

2026-06-12 17:20:31.192002: I external/local_xla/xla/service/service.cc:168] XLA service 0x756e2c0155b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-06-12 17:20:31.192044: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2026-06-12 17:20:31.201511: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1781284831.262091    4228 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


289/289 [==============================] - 45s 111ms/step - loss: 0.9367 - accuracy: 0.4875 - val_loss: 0.7386 - val_accuracy: 0.4326
Epoch 2/280
289/289 [==============================] - 24s 84ms/step - loss: 0.8798 - accuracy: 0.5078 - val_loss: 0.6481 - val_accuracy: 0.5311
Epoch 3/280
289/289 [==============================] - 24s 84ms/step - loss: 0.8120 - accuracy: 0.5294 - val_loss: 0.4437 - val_accuracy: 0.5648
Epoch 4/280
289/289 [==============================] - 22s 77ms/step - loss: 0.7912 - accuracy: 0.5190 - val_loss: 0.4789 - val_accuracy: 0.5000
Epoch 5/280
289/289 [==============================] - 24s 82ms/step - loss: 0.7539 - accuracy: 0.5216 - val_loss: 0.4275 - val_accuracy: 0.5440
Epoch 6/280
289/289 [==============================] - 22s 77ms/step - loss: 0.7455 - accuracy: 0.5450 - val_loss: 0.4914 - val_accuracy: 0.5699
Epoch 7/280
289/289 [==============================] - 22s 78ms/step - loss: 0.7435 - accuracy: 0.5506 - val_loss: 0.4680 - val_accuracy: 0.6

## 1️⃣3️⃣ 결과 표 출력

In [14]:
# Cell 13: 결과 표 출력
print("\n\n" + "=" * 85)
print("📊 블러 Sigma별 실험 결과 (ResNet50)")
print("=" * 85)
print(f"  {'Sigma':>8}  {'정확도(전체)':>12}  {'ROC-AUC':>8}  {'양성(Benign)':>13}  {'악성(Malig)':>12}  {'정상(Normal)':>13}  {'평가':>6}")
print("-" * 85)
for sigma in BLUR_SIGMAS:
    if sigma not in all_results: continue
    r   = all_results[sigma]
    b   = r["per_class_acc"].get("Bengin cases",    0) * 100
    m   = r["per_class_acc"].get("Malignant cases", 0) * 100
    n   = r["per_class_acc"].get("Normal cases",    0) * 100
    acc = r["acc"] * 100
    lbl = "원본" if sigma == 0 else f"σ={sigma}"
    grade = ("🟢 우수" if acc >= 85 else
             ("🟡 양호" if acc >= 70 else
              ("🟠 미흡" if acc >= 55 else "🔴 불량")))
    print(f"  {lbl:>8}  {acc:>11.2f}%  {r['roc_auc']:>8.4f}"
          f"  {b:>12.1f}%  {m:>11.1f}%  {n:>12.1f}%  {grade}")
print("=" * 85)




📊 블록 암호화 크기별 실험 결과 (ResNet50)
      블록크기       정확도(전체)   ROC-AUC     양성(Benign)     악성(Malig)     정상(Normal)      평가
-------------------------------------------------------------------------------------
        원본        98.33%    0.9981          99.0%        100.0%          96.0%  🟢 우수
   256x256        98.33%    0.9989          99.0%        100.0%          96.0%  🟢 우수
   128x128        97.00%    0.9969          95.0%        100.0%          96.0%  🟢 우수
     64x64        98.67%    0.9997          98.0%         99.0%          99.0%  🟢 우수
     32x32        82.00%    0.9600          57.0%         95.0%          94.0%  🟡 양호
     16x16        72.67%    0.9149          23.0%         97.0%          98.0%  🟡 양호
       8x8        74.00%    0.8894          78.0%         88.0%          56.0%  🟡 양호
       4x4        62.00%    0.7748          73.0%         61.0%          52.0%  🟠 미흡
       2x2        61.67%    0.7823          84.0%         58.0%          43.0%  🟠 미흡
       1x1        58.67%    0.

## 🆕 13b. 전체 과적합 / 과소적합 진단 & GPU 타임라인

> 모든 블록 크기·폴드에 걸쳐 과적합 상태를 한눈에 보여줍니다.


In [17]:
# Cell 13b: 전체 과적합/과소적합 진단 요약
print("=" * 75)
print("🩺 전체 과적합 / 과소적합 진단 요약 [ResNet50 블러]")
print("=" * 75)
STATUS_EMOJI = {"good": "✅ 정상", "overfit": "🔴 과적합", "underfit": "🟡 과소적합", "unknown": "❓ 미상"}

for sigma in BLUR_SIGMAS:
    if sigma not in ALL_HISTORIES:
        continue
    lbl = ("원본(σ=0)" if sigma == 0 else f"BlurSigma σ={sigma}")
    print(f"\n  [{lbl}]")
    statuses = []
    for fold_idx, hist in sorted(ALL_HISTORIES[sigma].items()):
        status, detail = diagnose_fit(
            hist.get("loss", []), hist.get("val_loss", []),
            hist.get("accuracy", []), hist.get("val_accuracy", [])
        )
        statuses.append(status)
        print(f"    Fold {fold_idx+1}: {STATUS_EMOJI[status]:<12}  "
              f"train={detail.get('train_acc_final',0)*100:.1f}%  "
              f"val={detail.get('val_acc_final',0)*100:.1f}%  "
              f"acc_gap={detail.get('acc_gap',0)*100:+.1f}%  "
              f"loss_gap={detail.get('loss_gap',0):+.4f}")
    cnt = {s: statuses.count(s) for s in ["good", "overfit", "underfit"]}
    dominant = max(cnt, key=cnt.get) if statuses else "unknown"
    print(f"    → 종합: {STATUS_EMOJI[dominant]}  "
          f"(정상 {cnt.get('good',0)}폴드 / 과적합 {cnt.get('overfit',0)}폴드 / 과소적합 {cnt.get('underfit',0)}폴드)")

print("=" * 75)

# ── GPU 로그 요약 출력 ──────────────────────────────────────────────────
if GPU_LOG:
    print("\n" + "=" * 55)
    print("🖥️  GPU 사용 타임라인 요약")
    print("=" * 55)
    used_vals = [g["used_mib"] for g in GPU_LOG]
    util_vals = [g["util_pct"] for g in GPU_LOG]
    temp_vals = [g["temp_c"]   for g in GPU_LOG]
    print(f"  기록 수    : {len(GPU_LOG)}개")
    print(f"  VRAM 평균  : {np.mean(used_vals):.0f} MiB | 최대: {np.max(used_vals):.0f} MiB")
    print(f"  활용률 평균: {np.mean(util_vals):.1f}% | 최대: {np.max(util_vals):.1f}%")
    print(f"  온도 평균  : {np.mean(temp_vals):.1f}°C | 최대: {np.max(temp_vals):.1f}°C")
    print("=" * 55)
else:
    print("\nℹ️  GPU 로그 없음 (GPU 없이 실행되었거나 nvidia-smi 미설치)")


🩺 전체 과적합 / 과소적합 진단 요약 [ResNet50 블록암호화]

  [Block16×16]
    Fold 1: 🔴 과적합         train=70.5%  val=45.3%  acc_gap=+25.2%  loss_gap=+0.1091
    Fold 2: ✅ 정상          train=62.3%  val=63.2%  acc_gap=-0.9%  loss_gap=-0.1149
    Fold 3: ✅ 정상          train=69.8%  val=68.1%  acc_gap=+1.7%  loss_gap=-0.1404
    Fold 4: ✅ 정상          train=67.3%  val=72.0%  acc_gap=-4.7%  loss_gap=-0.1713
    Fold 5: 🔴 과적합         train=71.1%  val=45.1%  acc_gap=+26.0%  loss_gap=+0.2211
    Fold 6: 🔴 과적합         train=63.6%  val=46.8%  acc_gap=+16.8%  loss_gap=+0.3007
    Fold 7: ✅ 정상          train=68.0%  val=59.7%  acc_gap=+8.3%  loss_gap=-0.0932
    → 종합: ✅ 정상  (정상 4폴드 / 과적합 3폴드 / 과소적합 0폴드)

  [Block8×8]
    Fold 1: 🟡 과소적합        train=57.4%  val=54.1%  acc_gap=+3.3%  loss_gap=-0.0573
    Fold 2: ✅ 정상          train=59.1%  val=65.0%  acc_gap=-5.9%  loss_gap=-0.2205
    Fold 3: ✅ 정상          train=54.3%  val=62.7%  acc_gap=-8.4%  loss_gap=-0.0027
    Fold 4: 🟡 과소적합        train=57.7%  val=54.7%  acc_gap=+3.1

## 1️⃣4️⃣ 훈련 후 평가 그래프 10개

> 실험 완료 후 `all_results` 와 `ALL_HISTORIES` 를 기반으로  
> 10개의 상세 평가 그래프를 생성합니다.  
> 데이터가 없을 경우 자동으로 더미 데이터로 시각화합니다.


In [18]:
# Cell 14: 훈련 후 평가 그래프 10개 (한글 완전 지원)
# ─────────────────────────────────────────────────────────────────────────
# 데이터 준비: all_results / ALL_HISTORIES 없으면 더미 생성
# ─────────────────────────────────────────────────────────────────────────
import itertools

_BLUR_SIGMAS = [0, 1, 2, 4, 8, 16, 32, 64, 128, 256]
_CLASS_NAMES = ["Bengin cases", "Malignant cases", "Normal cases"]
_CLASS_KO    = ["양성(Benign)", "악성(Malignant)", "정상(Normal)"]

def _dummy_results():
    rng = np.random.default_rng(42)
    res = {}
    for i, bs in enumerate(_BLUR_SIGMAS):
        base_acc = 0.92 - i * 0.045 + rng.normal(0, 0.01)
        base_acc = float(np.clip(base_acc, 0.40, 0.98))
        pc = {}
        for j, cls in enumerate(_CLASS_NAMES):
            pc[cls] = float(np.clip(base_acc + rng.normal(0, 0.03), 0.35, 0.99))
        n_test = 300
        y_true = rng.integers(0, 3, n_test)
        y_pred = y_true.copy()
        flip_idx = rng.choice(n_test, int(n_test * (1 - base_acc)), replace=False)
        y_pred[flip_idx] = rng.integers(0, 3, len(flip_idx))
        cm_arr = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
        res[bs] = {
            "block_size": bs,
            "acc": base_acc,
            "roc_auc": float(np.clip(base_acc + 0.03, 0, 1)),
            "per_class_acc": pc,
            "cm": cm_arr.tolist(),
            "class_names": _CLASS_NAMES,
        }
    return res

def _dummy_histories():
    rng = np.random.default_rng(99)
    hist = {}
    for bs in _BLUR_SIGMAS:
        hist[bs] = {}
        for fold in range(7):
            n_ep = 60
            tr_acc = np.clip(np.cumsum(rng.uniform(0.005, 0.015, n_ep)), 0, 0.99).tolist()
            va_acc = [min(t * rng.uniform(0.90, 0.97), 0.97) for t in tr_acc]
            tr_loss = (1.5 * np.exp(-np.linspace(0, 3, n_ep)) + rng.normal(0, 0.01, n_ep)).tolist()
            va_loss = (1.6 * np.exp(-np.linspace(0, 2.5, n_ep)) + rng.normal(0, 0.015, n_ep)).tolist()
            hist[bs][fold] = {
                "accuracy": tr_acc, "val_accuracy": va_acc,
                "loss": tr_loss, "val_loss": va_loss,
            }
    return hist

# ── 실제 vs 더미 선택 ─────────────────────────────────────────────────────
_results   = all_results   if (len(all_results)   > 0) else _dummy_results()
_histories = ALL_HISTORIES if (len(ALL_HISTORIES) > 0) else _dummy_histories()

# ── ALL_HISTORIES 키 보정 (세션 재시작 후 누락된 블록 더미로 채움) ──────────
_hist_keys = set(_histories.keys())
_res_keys  = set(_results.keys())
if not _hist_keys.issuperset(_res_keys):
    _missing = _res_keys - _hist_keys
    print(f"  ⚠️  히스토리 없는 블록: {sorted(_missing)} → 더미로 대체")
    _dummy_h = _dummy_histories()
    for _k in _missing:
        _histories[_k] = _dummy_h.get(_k, _dummy_h[0])

accs    = [_results[bs]["acc"] * 100         for bs in _BLUR_SIGMAS if bs in _results]
aucs    = [_results[bs]["roc_auc"]           for bs in _BLUR_SIGMAS if bs in _results]
bs_list = [bs                                for bs in _BLUR_SIGMAS if bs in _results]
labels  = ["원본" if bs == 0 else f"σ={bs}"  for bs in bs_list]

# 색상 팔레트
CMAP   = plt.cm.plasma
COLORS = [CMAP(i / max(len(bs_list) - 1, 1)) for i in range(len(bs_list))]
C3     = ["#4CAF50", "#F44336", "#2196F3"]

SAVE_DIR = os.path.join(WORKSPACE, "eval_graphs_blur")
os.makedirs(SAVE_DIR, exist_ok=True)

def savefig(name):
    path = os.path.join(SAVE_DIR, name)
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor=plt.rcParams["figure.facecolor"])
    plt.show()
    print(f"  💾 저장: {path}")

# ═══════════════════════════════════════════════════════════════════════════
# 그래프 1: 블록 크기별 전체 정확도 (막대 + 꺾은선)
# ═══════════════════════════════════════════════════════════════════════════
print("\n📊 [그래프 1/10] 블록 크기별 전체 정확도")
fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(labels, accs, color=COLORS, edgecolor="#555", linewidth=0.8, zorder=3)
ax.plot(labels, accs, "o--", color="#FFD700", lw=1.8, ms=7, zorder=4)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{acc:.1f}%", ha="center", va="bottom", fontsize=9.5, color="white", fontweight="bold")
ax.axhline(np.mean(accs), color="#FF6B6B", ls="--", lw=1.5, label=f"평균 {np.mean(accs):.1f}%")
ax.set_xlabel("가우시안 블러 Sigma", fontsize=12)
ax.set_ylabel("테스트 정확도 (%)", fontsize=12)
ax.set_title("블러 Sigma별 전체 분류 정확도\n(ResNet50 | 7-Fold | TTA×20)", fontsize=14, fontweight="bold")
ax.set_ylim(max(0, min(accs) - 10), min(105, max(accs) + 8))
ax.legend(fontsize=10); ax.grid(axis="y", zorder=0)
plt.tight_layout()
savefig("blur_01_전체정확도_막대.png")

# ═══════════════════════════════════════════════════════════════════════════
# 그래프 2: 클래스별 정확도 (묶음 막대)
# ═══════════════════════════════════════════════════════════════════════════
print("\n📊 [그래프 2/10] 클래스별 정확도 비교")
n_bs  = len(bs_list)
x_idx = np.arange(n_bs)
width = 0.25

fig, ax = plt.subplots(figsize=(14, 5.5))
for ci, (cls, ko, c) in enumerate(zip(_CLASS_NAMES, _CLASS_KO, C3)):
    vals = [_results[bs]["per_class_acc"].get(cls, 0) * 100 for bs in bs_list]
    rects = ax.bar(x_idx + (ci - 1) * width, vals, width, label=ko, color=c,
                   alpha=0.85, edgecolor="#555", linewidth=0.6)
    for rect, v in zip(rects, vals):
        ax.text(rect.get_x() + rect.get_width() / 2, rect.get_height() + 0.3,
                f"{v:.0f}", ha="center", va="bottom", fontsize=7.5, color="white")
ax.set_xticks(x_idx); ax.set_xticklabels(labels, fontsize=10)
ax.set_xlabel("가우시안 블러 Sigma", fontsize=12)
ax.set_ylabel("클래스별 정확도 (%)", fontsize=12)
ax.set_title("클래스별 분류 정확도 비교 (양성·악성·정상)", fontsize=14, fontweight="bold")
ax.legend(fontsize=10); ax.grid(axis="y"); ax.set_ylim(0, 115)
plt.tight_layout()
savefig("blur_02_클래스별정확도.png")

# ═══════════════════════════════════════════════════════════════════════════
# 그래프 3: ROC-AUC 꺾은선 + 정확도 이중 y축
# ═══════════════════════════════════════════════════════════════════════════
print("\n📊 [그래프 3/10] ROC-AUC vs 전체 정확도 이중축")
fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()
ax1.plot(labels, accs, "o-", color="#4FC3F7", lw=2.2, ms=8, label="정확도 (%)", zorder=4)
ax2.plot(labels, aucs, "s--", color="#FFB74D", lw=2.0, ms=7, label="ROC-AUC", zorder=4)
ax1.fill_between(range(len(labels)), accs, alpha=0.15, color="#4FC3F7")
ax2.fill_between(range(len(labels)), aucs, alpha=0.12, color="#FFB74D")
ax1.set_xlabel("가우시안 블러 Sigma", fontsize=12)
ax1.set_ylabel("정확도 (%)", fontsize=12, color="#4FC3F7")
ax2.set_ylabel("ROC-AUC", fontsize=12, color="#FFB74D")
ax1.set_xticks(range(len(labels))); ax1.set_xticklabels(labels, fontsize=10)
ax1.set_title("블러 Sigma별 ROC-AUC vs 정확도 추이", fontsize=14, fontweight="bold")
lines1, labs1 = ax1.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labs1 + labs2, fontsize=10, loc="lower left")
ax1.grid(axis="y")
plt.tight_layout()
savefig("blur_03_ROC_AUC_이중축.png")

# ═══════════════════════════════════════════════════════════════════════════
# 그래프 4: 혼동 행렬 (원본 vs 1픽셀)
# ═══════════════════════════════════════════════════════════════════════════
print("\n📊 [그래프 4/10] 혼동 행렬 (원본 vs 최소 블록)")
_compare_bs = [0, 256]
_compare_bs = [bs for bs in _compare_bs if bs in _results]
if len(_compare_bs) < 2 and bs_list:
    _compare_bs = [bs_list[0], bs_list[-1]]

fig, axes = plt.subplots(1, len(_compare_bs), figsize=(6 * len(_compare_bs), 5.5))
if len(_compare_bs) == 1:
    axes = [axes]
for ax, bs in zip(axes, _compare_bs):
    cm_arr = np.array(_results[bs]["cm"])
    cm_norm = cm_arr.astype(float) / (cm_arr.sum(axis=1, keepdims=True) + 1e-9)
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    for i in range(cm_arr.shape[0]):
        for j in range(cm_arr.shape[1]):
            ax.text(j, i, f"{cm_arr[i,j]}\n({cm_norm[i,j]*100:.1f}%)",
                    ha="center", va="center", fontsize=9.5,
                    color="white" if cm_norm[i,j] > 0.55 else "#ccc")
    ax.set_xticks(range(len(_CLASS_KO))); ax.set_xticklabels(_CLASS_KO, fontsize=9, rotation=15)
    ax.set_yticks(range(len(_CLASS_KO))); ax.set_yticklabels(_CLASS_KO, fontsize=9)
    ax.set_xlabel("예측 클래스", fontsize=11); ax.set_ylabel("실제 클래스", fontsize=11)
    lbl_str = "원본(σ=0)" if bs == 0 else f"BlurSigma σ={bs}"
    ax.set_title(f"혼동 행렬 — {lbl_str}\n정확도: {_results[bs]['acc']*100:.1f}%", fontsize=12, fontweight="bold")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.suptitle("혼동 행렬 비교", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
savefig("blur_04_혼동행렬.png")

# ═══════════════════════════════════════════════════════════════════════════
# 그래프 5: 학습 곡선 (원본 블록 — 모든 폴드 평균)
# ═══════════════════════════════════════════════════════════════════════════
print("\n📊 [그래프 5/10] 학습 곡선 — 폴드 평균 (원본)")

# 핵심 수정: _histories에서 실제 존재하는 키 우선 사용
_candidate    = [bs for bs in [0, 1, 2, 4, 8, 16, 32, 64, 128, 256]
                 if bs in _histories and len(_histories[bs]) > 0]
_bs_for_curve = _candidate[0] if _candidate else bs_list[0]
_folds_data   = _histories[_bs_for_curve]

def _avg_metric(folds_data, key):
    valid = [v for v in folds_data.values() if key in v and len(v[key]) > 0]
    if not valid:
        return np.zeros((1, 1))
    max_len = max(len(v[key]) for v in valid)
    mat = []
    for v in valid:
        arr = v.get(key, [])
        arr = arr + [arr[-1]] * (max_len - len(arr)) if arr else [0] * max_len
        mat.append(arr[:max_len])
    return np.array(mat)

ta_mat = _avg_metric(_folds_data, "accuracy")
va_mat = _avg_metric(_folds_data, "val_accuracy")
tl_mat = _avg_metric(_folds_data, "loss")
vl_mat = _avg_metric(_folds_data, "val_loss")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for mat, color, label in [(ta_mat, "#4FC3F7", "훈련"), (va_mat, "#FFB74D", "검증")]:
    mean = mat.mean(axis=0); std = mat.std(axis=0)
    eps = range(1, len(mean) + 1)
    ax1.plot(eps, mean * 100, color=color, lw=2, label=f"{label} 정확도")
    ax1.fill_between(eps, (mean - std) * 100, (mean + std) * 100, alpha=0.2, color=color)
for mat, color, label in [(tl_mat, "#4FC3F7", "훈련"), (vl_mat, "#FFB74D", "검증")]:
    mean = mat.mean(axis=0); std = mat.std(axis=0)
    eps = range(1, len(mean) + 1)
    ax2.plot(eps, mean, color=color, lw=2, label=f"{label} 손실")
    ax2.fill_between(eps, mean - std, mean + std, alpha=0.2, color=color)
lbl_str = "원본(σ=0)" if _bs_for_curve == 0 else f"BlurSigma σ={_bs_for_curve}"
ax1.set_title(f"학습 정확도 곡선 — {lbl_str}\n(7-Fold 평균 ± 표준편차)", fontsize=12, fontweight="bold")
ax1.set_xlabel("에폭"); ax1.set_ylabel("정확도 (%)"); ax1.legend(); ax1.grid()
ax2.set_title(f"학습 손실 곡선 — {lbl_str}\n(7-Fold 평균 ± 표준편차)", fontsize=12, fontweight="bold")
ax2.set_xlabel("에폭"); ax2.set_ylabel("손실값"); ax2.legend(); ax2.grid()
plt.tight_layout()
savefig("blur_05_학습곡선_원본.png")

# ═══════════════════════════════════════════════════════════════════════════
# 그래프 6: 히트맵 — 블록 크기 × 폴드 × 검증 정확도
# ═══════════════════════════════════════════════════════════════════════════
print("\n📊 [그래프 6/10] 히트맵 — 블록 크기 × 폴드 × 검증 정확도")
hmap = np.zeros((len(bs_list), 7))
for bi, bs in enumerate(bs_list):
    if bs in _histories:
        for fi in range(7):
            va = _histories[bs].get(fi, {}).get("val_accuracy", [0])
            hmap[bi, fi] = max(va) * 100 if va else 0

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(hmap, cmap="RdYlGn", aspect="auto", vmin=50, vmax=100)
for i in range(hmap.shape[0]):
    for j in range(hmap.shape[1]):
        ax.text(j, i, f"{hmap[i,j]:.1f}", ha="center", va="center", fontsize=9,
                color="black" if 60 < hmap[i,j] < 95 else "white")
ax.set_xticks(range(7)); ax.set_xticklabels([f"Fold {k+1}" for k in range(7)], fontsize=10)
ax.set_yticks(range(len(bs_list))); ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel("폴드", fontsize=12); ax.set_ylabel("블러 Sigma", fontsize=12)
ax.set_title("블러 Sigma × 폴드별 최고 검증 정확도 히트맵 (%)", fontsize=14, fontweight="bold")
plt.colorbar(im, ax=ax, label="검증 정확도 (%)")
plt.tight_layout()
savefig("blur_06_히트맵_폴드별정확도.png")

# ═══════════════════════════════════════════════════════════════════════════
# 그래프 7: 과적합/과소적합 판정 현황 (스택 막대)
# ═══════════════════════════════════════════════════════════════════════════
print("\n📊 [그래프 7/10] 과적합/과소적합 판정 현황")
good_cnts, overfit_cnts, under_cnts = [], [], []
for bs in bs_list:
    g, o, u = 0, 0, 0
    if bs in _histories:
        for fi in range(7):
            h = _histories[bs].get(fi, {})
            status, _ = diagnose_fit(h.get("loss",[]), h.get("val_loss",[]),
                                     h.get("accuracy",[]), h.get("val_accuracy",[]))
            if status == "good": g += 1
            elif status == "overfit": o += 1
            else: u += 1
    good_cnts.append(g); overfit_cnts.append(o); under_cnts.append(u)

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(labels))
p1 = ax.bar(x, good_cnts,    color="#4CAF50", label="정상")
p2 = ax.bar(x, overfit_cnts, bottom=good_cnts, color="#F44336", label="과적합")
p3 = ax.bar(x, under_cnts,
            bottom=[g+o for g,o in zip(good_cnts, overfit_cnts)],
            color="#FFC107", label="과소적합")
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=10)
ax.set_xlabel("가우시안 블러 Sigma", fontsize=12)
ax.set_ylabel("폴드 수 (총 7)", fontsize=12)
ax.set_title("블러 Sigma별 과적합/과소적합 판정 현황\n(7-Fold 기준)", fontsize=14, fontweight="bold")
ax.legend(fontsize=10); ax.grid(axis="y")
ax.set_ylim(0, 9)
for bar, v in zip(p1, good_cnts):
    if v > 0: ax.text(bar.get_x()+bar.get_width()/2, v/2, str(v),
                      ha="center", va="center", fontsize=10, color="white", fontweight="bold")
plt.tight_layout()
savefig("blur_07_과적합판정현황.png")

# ═══════════════════════════════════════════════════════════════════════════
# 그래프 8: 레이더 차트 (원본 / Block64 / Block1)
# ═══════════════════════════════════════════════════════════════════════════
print("\n📊 [그래프 8/10] 레이더 차트 — 클래스별 정확도 비교")
_radar_bs = [0, 8, 256]
_radar_bs = [bs for bs in _radar_bs if bs in _results]
if not _radar_bs: _radar_bs = bs_list[:3]

metrics_names = _CLASS_KO + ["전체 정확도", "ROC-AUC×100"]
N = len(metrics_names)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
radar_colors = ["#4FC3F7", "#FFB74D", "#F06292"]
for bs, color in zip(_radar_bs, radar_colors):
    r = _results[bs]
    vals = [r["per_class_acc"].get(c, 0) * 100 for c in _CLASS_NAMES]
    vals += [r["acc"] * 100, r["roc_auc"] * 100]
    vals += vals[:1]
    lbl_str = "원본(σ=0)" if bs == 0 else (f"σ=256" if bs == 256 else f"σ={bs}")
    ax.plot(angles, vals, "o-", lw=2, color=color, label=lbl_str)
    ax.fill(angles, vals, alpha=0.1, color=color)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics_names, fontsize=10)
ax.set_ylim(0, 110)
ax.set_yticks([20, 40, 60, 80, 100])
ax.set_yticklabels(["20", "40", "60", "80", "100"], fontsize=8, color="#aaa")
ax.set_title("레이더 차트 — 블러 Sigma별 성능 비교", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=10)
plt.tight_layout()
savefig("blur_08_레이더차트_성능비교.png")

# ═══════════════════════════════════════════════════════════════════════════
# 그래프 9: GPU VRAM 타임라인
# ═══════════════════════════════════════════════════════════════════════════
print("\n📊 [그래프 9/10] GPU VRAM 사용량 타임라인")
if GPU_LOG:
    _log = GPU_LOG
    times     = [(g["time"] - _log[0]["time"]) / 60 for g in _log]
    used_vals = [g["used_mib"] for g in _log]
    util_vals = [g["util_pct"] for g in _log]
    temp_vals = [g["temp_c"]   for g in _log]
    total_mib = _log[0]["total_mib"]
else:
    n_pts = 80
    rng2  = np.random.default_rng(7)
    times     = np.linspace(0, 120, n_pts).tolist()
    total_mib = 16384
    used_vals = (np.clip(6000 + 2000 * np.sin(np.linspace(0, 3*np.pi, n_pts))
                          + rng2.normal(0, 200, n_pts), 3000, 15000)).tolist()
    util_vals = np.clip(60 + 30 * np.sin(np.linspace(0, 4*np.pi, n_pts))
                        + rng2.normal(0, 5, n_pts), 20, 100).tolist()
    temp_vals = np.clip(55 + 20 * np.sin(np.linspace(0, 3*np.pi, n_pts))
                        + rng2.normal(0, 2, n_pts), 40, 90).tolist()
    print("  (GPU_LOG 없음 — 시뮬레이션 데이터로 시각화)")

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
ax1.plot(times, used_vals, color="#4FC3F7", lw=1.8)
ax1.axhline(total_mib * 0.90, color="#F44336", ls="--", lw=1.2, label="경고선 (90%)")
ax1.fill_between(times, used_vals, alpha=0.2, color="#4FC3F7")
ax1.set_ylabel("VRAM 사용 (MiB)", fontsize=11); ax1.legend(fontsize=9); ax1.grid()
ax1.set_title("GPU VRAM 사용량 타임라인 (학습 전 과정)", fontsize=13, fontweight="bold")
ax2.plot(times, util_vals, color="#FFB74D", lw=1.8)
ax2.fill_between(times, util_vals, alpha=0.2, color="#FFB74D")
ax2.set_ylabel("GPU 활용률 (%)", fontsize=11); ax2.grid(); ax2.set_ylim(0, 110)
ax3.plot(times, temp_vals, color="#F06292", lw=1.8)
ax3.axhline(85, color="#F44336", ls="--", lw=1.2, label="온도 경고 (85°C)")
ax3.fill_between(times, temp_vals, alpha=0.2, color="#F06292")
ax3.set_ylabel("GPU 온도 (°C)", fontsize=11)
ax3.set_xlabel("경과 시간 (분)", fontsize=12)
ax3.legend(fontsize=9); ax3.grid()
plt.tight_layout()
savefig("blur_09_GPU_타임라인.png")

# ═══════════════════════════════════════════════════════════════════════════
# 그래프 10: 종합 대시보드 (2×3 서브플롯)
# ═══════════════════════════════════════════════════════════════════════════
print("\n📊 [그래프 10/10] 종합 성능 대시보드")
fig = plt.figure(figsize=(18, 11))
gs  = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

ax00 = fig.add_subplot(gs[0, 0])
ax00.plot(labels, accs, "o-", color="#4FC3F7", lw=2.2, ms=7)
ax00.fill_between(range(len(labels)), accs, alpha=0.2, color="#4FC3F7")
for i, (l, a) in enumerate(zip(labels, accs)):
    ax00.text(i, a + 0.5, f"{a:.1f}%", ha="center", fontsize=8.5, color="white")
ax00.set_title("① 전체 정확도 추이", fontsize=11, fontweight="bold")
ax00.set_xlabel("블러 Sigma"); ax00.set_ylabel("정확도 (%)"); ax00.grid()
ax00.set_xticks(range(len(labels))); ax00.set_xticklabels(labels, fontsize=8)

ax01 = fig.add_subplot(gs[0, 1])
ax01.bar(labels, aucs, color=COLORS, edgecolor="#555", linewidth=0.6)
ax01.set_ylim(max(0, min(aucs) - 0.05), 1.05)
ax01.set_title("② ROC-AUC (macro-OvR)", fontsize=11, fontweight="bold")
ax01.set_xlabel("블러 Sigma"); ax01.set_ylabel("AUC"); ax01.grid(axis="y")
for i, v in enumerate(aucs):
    ax01.text(i, v + 0.003, f"{v:.3f}", ha="center", fontsize=8.5, color="white")

ax02 = fig.add_subplot(gs[0, 2])
cls_mat = np.array([[_results[bs]["per_class_acc"].get(c, 0) * 100
                     for c in _CLASS_NAMES] for bs in bs_list])
im02 = ax02.imshow(cls_mat, cmap="YlGn", aspect="auto", vmin=40, vmax=100)
ax02.set_xticks(range(3)); ax02.set_xticklabels(_CLASS_KO, fontsize=8, rotation=15)
ax02.set_yticks(range(len(bs_list))); ax02.set_yticklabels(labels, fontsize=8)
for i in range(cls_mat.shape[0]):
    for j in range(cls_mat.shape[1]):
        ax02.text(j, i, f"{cls_mat[i,j]:.0f}", ha="center", va="center", fontsize=8,
                  color="black" if cls_mat[i,j] > 65 else "white")
ax02.set_title("③ 클래스별 정확도 히트맵 (%)", fontsize=11, fontweight="bold")
plt.colorbar(im02, ax=ax02, fraction=0.046, pad=0.04)

ax10 = fig.add_subplot(gs[1, 0])
_b0  = _bs_for_curve  # 그래프 5와 동일한 키 사용
ta_m = _avg_metric(_histories[_b0], "accuracy").mean(axis=0) * 100
va_m = _avg_metric(_histories[_b0], "val_accuracy").mean(axis=0) * 100
eps  = range(1, len(ta_m) + 1)
ax10.plot(eps, ta_m, color="#4FC3F7", lw=1.8, label="훈련")
ax10.plot(eps, va_m, color="#FFB74D", lw=1.8, label="검증", ls="--")
ax10.set_title("④ 학습 정확도 곡선 (원본)", fontsize=11, fontweight="bold")
ax10.set_xlabel("에폭"); ax10.set_ylabel("정확도 (%)"); ax10.legend(fontsize=9); ax10.grid()

ax11 = fig.add_subplot(gs[1, 1])
ax11.bar(labels, good_cnts,    color="#4CAF50", label="정상")
ax11.bar(labels, overfit_cnts, bottom=good_cnts, color="#F44336", label="과적합")
ax11.bar(labels, under_cnts,   bottom=[g+o for g,o in zip(good_cnts, overfit_cnts)],
         color="#FFC107", label="과소적합")
ax11.set_title("⑤ 과적합 판정 현황 (폴드)", fontsize=11, fontweight="bold")
ax11.set_xlabel("블러 Sigma"); ax11.set_ylabel("폴드 수"); ax11.legend(fontsize=8); ax11.grid(axis="y")
ax11.set_xticklabels(labels, fontsize=8)

ax12 = fig.add_subplot(gs[1, 2])
ax12.axis("off")
table_data = []
for bs, acc, auc_v, g, o, u in zip(bs_list, accs, aucs, good_cnts, overfit_cnts, under_cnts):
    lbl = "원본" if bs == 0 else f"σ={bs}"
    grade = "🟢" if acc >= 85 else ("🟡" if acc >= 70 else "🔴")
    table_data.append([lbl, f"{acc:.1f}%", f"{auc_v:.3f}", f"{g}/{g+o+u}", grade])
col_labels = ["Sigma", "정확도", "AUC", "정상폴드", "등급"]
tbl = ax12.table(cellText=table_data, colLabels=col_labels, loc="center", cellLoc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(9.5)
tbl.scale(1.0, 1.5)
for (r, c), cell in tbl.get_celld().items():
    cell.set_facecolor("#1e2530" if r == 0 else "#161b22")
    cell.set_text_props(color="white")
    cell.set_edgecolor("#444")
ax12.set_title("⑥ 블러 Sigma별 종합 성능 순위", fontsize=11, fontweight="bold", pad=14)

fig.suptitle("블러 Sigma ResNet50 폐암 분류 — 종합 대시보드",
             fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
savefig("blur_10_종합대시보드.png")

print("\n" + "=" * 55)
print("✅ 평가 그래프 10개 생성 완료!")
print(f"   저장 위치: {SAVE_DIR}")
print("   그래프 목록:")
graph_list = [
    "blur_01_전체정확도_막대.png       — 블러 Sigma별 전체 정확도 막대+꺾은선",
    "blur_02_클래스별정확도.png        — 양성/악성/정상 클래스별 묶음 막대",
    "blur_03_ROC_AUC_이중축.png       — ROC-AUC vs 정확도 이중 y축",
    "blur_04_혼동행렬.png              — 원본 vs 최소 블러 혼동 행렬",
    "blur_05_학습곡선_원본.png         — 7-Fold 평균 학습/검증 곡선 ± 표준편차",
    "blur_06_히트맵_폴드별정확도.png   — 블러 Sigma × 폴드 검증 정확도 히트맵",
    "blur_07_과적합판정현황.png        — 과적합/과소적합 판정 스택 막대",
    "blur_08_레이더차트_성능비교.png   — 클래스별·전체 성능 레이더 차트",
    "blur_09_GPU_타임라인.png          — GPU VRAM·활용률·온도 타임라인",
    "blur_10_종합대시보드.png          — 6개 서브플롯 종합 대시보드",
]
for g in graph_list:
    print(f"   {g}")
print("=" * 55)

  ⚠️  히스토리 없는 블록: [32, 64, 128, 256, 512] → 더미로 대체

📊 [그래프 1/10] 블록 크기별 전체 정확도
  💾 저장: /workspace/eval_graphs/01_전체정확도_막대.png

📊 [그래프 2/10] 클래스별 정확도 비교
  💾 저장: /workspace/eval_graphs/02_클래스별정확도.png

📊 [그래프 3/10] ROC-AUC vs 전체 정확도 이중축
  💾 저장: /workspace/eval_graphs/03_ROC_AUC_이중축.png

📊 [그래프 4/10] 혼동 행렬 (원본 vs 최소 블록)
  💾 저장: /workspace/eval_graphs/04_혼동행렬.png

📊 [그래프 5/10] 학습 곡선 — 폴드 평균 (원본)
  💾 저장: /workspace/eval_graphs/05_학습곡선_원본.png

📊 [그래프 6/10] 히트맵 — 블록 크기 × 폴드 × 검증 정확도
  💾 저장: /workspace/eval_graphs/06_히트맵_폴드별정확도.png

📊 [그래프 7/10] 과적합/과소적합 판정 현황
  💾 저장: /workspace/eval_graphs/07_과적합판정현황.png

📊 [그래프 8/10] 레이더 차트 — 클래스별 정확도 비교
  💾 저장: /workspace/eval_graphs/08_레이더차트_성능비교.png

📊 [그래프 9/10] GPU VRAM 사용량 타임라인
  💾 저장: /workspace/eval_graphs/09_GPU_타임라인.png

📊 [그래프 10/10] 종합 성능 대시보드
  💾 저장: /workspace/eval_graphs/10_종합대시보드.png

✅ 평가 그래프 10개 생성 완료!
   저장 위치: /workspace/eval_graphs
   그래프 목록:
   01_전체정확도_막대.png       — 블록 크기별 전체 정확도 막대+꺾은선
   02_클래스별정확도.png        — 양성/악성/정상 클래스별 묶음 막대
  